# ⚡ FastAPI Methods — Complete Notebook

> **FastAPI** is a modern, high-performance Python web framework for building APIs with automatic docs, type safety, and async support.  
> This notebook covers every major FastAPI concept with runnable code examples and best practices.

---

## 📋 Table of Contents
1. [Installation & Setup](#1-installation--setup)
2. [Basic App & Routing](#2-basic-app--routing)
3. [Path & Query Parameters](#3-path--query-parameters)
4. [Request Body & Pydantic Models](#4-request-body--pydantic-models)
5. [Response Models & Status Codes](#5-response-models--status-codes)
6. [Headers, Cookies & Form Data](#6-headers-cookies--form-data)
7. [File Uploads](#7-file-uploads)
8. [Dependency Injection](#8-dependency-injection)
9. [Authentication & Security](#9-authentication--security)
10. [Database Integration (SQLAlchemy)](#10-database-integration-sqlalchemy)
11. [Async Endpoints](#11-async-endpoints)
12. [Background Tasks](#12-background-tasks)
13. [Middleware](#13-middleware)
14. [CORS](#14-cors)
15. [WebSockets](#15-websockets)
16. [Error Handling & Custom Exceptions](#16-error-handling--custom-exceptions)
17. [APIRouter & App Structure](#17-apirouter--app-structure)
18. [Testing with TestClient](#18-testing-with-testclient)
19. [Lifespan Events (Startup/Shutdown)](#19-lifespan-events-startupshutdown)
20. [Caching & Rate Limiting](#20-caching--rate-limiting)
21. [Deployment & Configuration](#21-deployment--configuration)
22. [Async SQLAlchemy](#22-async-sqlalchemy)
23. [Pagination Patterns](#23-pagination-patterns)
24. [Server-Sent Events (SSE)](#24-server-sent-events-sse)
25. [Static Files & Jinja2 Templates](#25-static-files--jinja2-templates)
26. [OpenAPI Customization](#26-openapi-customization)
27. [Celery Task Queues](#27-celery-task-queues)
28. [Prometheus Metrics & OpenTelemetry](#28-prometheus-metrics--opentelemetry)
29. [GraphQL with Strawberry](#29-graphql-with-strawberry)
30. [Docker Compose & Nginx](#30-docker-compose--nginx)
31. [gRPC with FastAPI](#31-grpc-with-fastapi)
32. [Multi-tenancy Patterns](#32-multi-tenancy-patterns)
12. [Background Tasks](#12-background-tasks)
13. [Middleware](#13-middleware)
14. [CORS](#14-cors)
15. [WebSockets](#15-websockets)
16. [Error Handling & Custom Exceptions](#16-error-handling--custom-exceptions)
17. [APIRouter & App Structure](#17-apirouter--app-structure)
18. [Testing with TestClient](#18-testing-with-testclient)
19. [Lifespan Events (Startup/Shutdown)](#19-lifespan-events-startupshutdown)
20. [Caching & Rate Limiting](#20-caching--rate-limiting)
21. [Deployment & Configuration](#21-deployment--configuration)

---
## 1. Installation & Setup

In [ ]:
# Install FastAPI and all common dependencies
!pip install fastapi uvicorn[standard] httpx python-multipart python-jose[cryptography] \
             passlib[bcrypt] sqlalchemy aiosqlite pydantic-settings slowapi redis

In [ ]:
# ── Notebook helper: run FastAPI app in background thread ─────
# Since Jupyter blocks, we use a thread + httpx to test endpoints
import threading
import time
import httpx
import uvicorn

def run_app(app, port=8000):
    """Start a FastAPI app in a background thread."""
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="error")
    server = uvicorn.Server(config)
    t = threading.Thread(target=server.run, daemon=True)
    t.start()
    time.sleep(1)  # wait for startup
    return server

print("Helper ready. We'll use run_app(app, port=XXXX) + httpx to test each section.")

---
## 2. Basic App & Routing

A FastAPI app is created with `FastAPI()`. Routes are defined using HTTP method decorators.

In [ ]:
from fastapi import FastAPI

app = FastAPI(
    title="My API",
    description="A demo FastAPI application",
    version="1.0.0",
    docs_url="/docs",        # Swagger UI
    redoc_url="/redoc",      # ReDoc UI
    openapi_url="/openapi.json"
)

# ── GET ───────────────────────────────────────────────────────
@app.get("/")
def root():
    return {"message": "Welcome to FastAPI!"}

# ── POST ──────────────────────────────────────────────────────
@app.post("/items")
def create_item(name: str):
    return {"created": name}

# ── PUT ───────────────────────────────────────────────────────
@app.put("/items/{item_id}")
def update_item(item_id: int, name: str):
    return {"updated": item_id, "name": name}

# ── PATCH ─────────────────────────────────────────────────────
@app.patch("/items/{item_id}")
def partial_update(item_id: int):
    return {"patched": item_id}

# ── DELETE ────────────────────────────────────────────────────
@app.delete("/items/{item_id}")
def delete_item(item_id: int):
    return {"deleted": item_id}

# ── HEAD / OPTIONS ────────────────────────────────────────────
@app.head("/health")
def health_head():
    return {}  # HEAD returns no body

@app.options("/items")
def items_options():
    return {"allow": "GET, POST, OPTIONS"}

In [ ]:
# ── Test the basic app ────────────────────────────────────────
server = run_app(app, port=8000)

with httpx.Client(base_url="http://127.0.0.1:8000") as client:
    print("GET  /      :", client.get("/").json())
    print("POST /items :", client.post("/items?name=laptop").json())
    print("PUT  /items/1:", client.put("/items/1?name=phone").json())
    print("DEL  /items/1:", client.delete("/items/1").json())

---
## 3. Path & Query Parameters

FastAPI automatically parses and validates URL path segments and query string parameters.

In [ ]:
from fastapi import FastAPI, Path, Query
from typing import Optional, List
from enum import Enum

app2 = FastAPI()

class Category(str, Enum):
    electronics = "electronics"
    clothing    = "clothing"
    food        = "food"

# ── Path parameters with validation ──────────────────────────
@app2.get("/users/{user_id}")
def get_user(
    user_id: int = Path(..., ge=1, le=9999, description="User ID between 1 and 9999")
):
    return {"user_id": user_id}

# ── Enum path parameter ───────────────────────────────────────
@app2.get("/products/category/{category}")
def get_by_category(category: Category):
    return {"category": category.value}

# ── Query parameters with defaults & validation ───────────────
@app2.get("/products")
def list_products(
    page:     int            = Query(default=1,    ge=1,   description="Page number"),
    limit:    int            = Query(default=10,   ge=1,   le=100),
    search:   Optional[str]  = Query(default=None, min_length=2, max_length=50),
    in_stock: bool           = Query(default=True),
    tags:     List[str]      = Query(default=[])   # ?tags=a&tags=b
):
    return {
        "page": page, "limit": limit,
        "search": search, "in_stock": in_stock, "tags": tags
    }

# ── Path + Query together ─────────────────────────────────────
@app2.get("/users/{user_id}/orders")
def get_user_orders(
    user_id: int,
    status: Optional[str] = None,
    limit:  int = 10
):
    return {"user_id": user_id, "status": status, "limit": limit}

In [ ]:
server2 = run_app(app2, port=8001)

with httpx.Client(base_url="http://127.0.0.1:8001") as c:
    print(c.get("/users/42").json())
    print(c.get("/products/category/electronics").json())
    print(c.get("/products", params={"page": 2, "limit": 5, "search": "phone", "tags": ["new", "sale"]}).json())
    print(c.get("/users/1/orders", params={"status": "shipped", "limit": 3}).json())
    # Validation error:
    r = c.get("/users/0")  # user_id must be >= 1
    print("Validation error:", r.status_code, r.json()["detail"][0]["msg"])

---
## 4. Request Body & Pydantic Models

Use Pydantic models to define, validate, and document request bodies.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Optional, List
from datetime import datetime

app3 = FastAPI()

# ── Basic model ───────────────────────────────────────────────
class Address(BaseModel):
    street: str
    city:   str
    zip:    str = Field(..., pattern=r"^\d{5}$", description="5-digit ZIP code")

class UserCreate(BaseModel):
    name:     str     = Field(...,  min_length=2, max_length=50)
    email:    str     = Field(...,  description="Valid email address")
    age:      int     = Field(...,  ge=0, le=120)
    score:    float   = Field(0.0, ge=0.0, le=100.0)
    tags:     List[str] = Field(default=[])
    address:  Optional[Address] = None
    is_active: bool  = True

    @field_validator("email")
    @classmethod
    def email_must_have_at(cls, v):
        if "@" not in v:
            raise ValueError("Not a valid email")
        return v.lower()

    @field_validator("name")
    @classmethod
    def name_must_be_alpha(cls, v):
        if not v.replace(" ", "").isalpha():
            raise ValueError("Name must contain only letters")
        return v.title()

class UserResponse(BaseModel):
    id:         int
    name:       str
    email:      str
    created_at: datetime

@app3.post("/users", response_model=UserResponse)
def create_user(user: UserCreate):
    return UserResponse(
        id=1, name=user.name,
        email=user.email, created_at=datetime.utcnow()
    )

# ── Multiple body parameters ──────────────────────────────────
class Item(BaseModel):
    name:  str
    price: float = Field(..., gt=0)

class Supplier(BaseModel):
    company: str
    country: str

@app3.post("/item-with-supplier")
def create_with_supplier(item: Item, supplier: Supplier):
    return {"item": item.model_dump(), "supplier": supplier.model_dump()}

# ── Partial update with Optional fields ──────────────────────
class UserUpdate(BaseModel):
    name:  Optional[str] = None
    email: Optional[str] = None
    age:   Optional[int] = None

@app3.patch("/users/{user_id}")
def update_user(user_id: int, update: UserUpdate):
    changed = {k: v for k, v in update.model_dump().items() if v is not None}
    return {"user_id": user_id, "updated_fields": changed}

In [ ]:
import json

server3 = run_app(app3, port=8002)

with httpx.Client(base_url="http://127.0.0.1:8002") as c:
    # Valid user creation
    r = c.post("/users", json={
        "name": "Alice Smith", "email": "Alice@Example.com",
        "age": 30, "tags": ["admin"],
        "address": {"street": "123 Main St", "city": "NY", "zip": "10001"}
    })
    print("Created user:", r.json())

    # Validation error
    r = c.post("/users", json={"name": "Bob123", "email": "notvalid", "age": -1})
    print("\nValidation errors:")
    for err in r.json()["detail"]:
        print(f"  {err['loc']}: {err['msg']}")

    # Partial update
    r = c.patch("/users/1", json={"email": "newemail@example.com"})
    print("\nPartial update:", r.json())

---
## 5. Response Models & Status Codes

Control what gets returned and with which HTTP status code.

In [ ]:
from fastapi import FastAPI, status
from fastapi.responses import JSONResponse, HTMLResponse, PlainTextResponse, RedirectResponse, FileResponse
from pydantic import BaseModel
from typing import Optional, List

app4 = FastAPI()

class Product(BaseModel):
    id:          int
    name:        str
    price:       float
    description: Optional[str] = None
    secret_key:  str = "hidden"  # we'll exclude this

class ProductOut(BaseModel):
    id:    int
    name:  str
    price: float

# ── response_model filters output fields ─────────────────────
@app4.get("/products/{id}", response_model=ProductOut)
def get_product(id: int):
    return Product(id=id, name="Laptop", price=999.99, secret_key="abc123")  # secret_key filtered out

# ── Custom status codes ───────────────────────────────────────
@app4.post("/products", response_model=ProductOut, status_code=status.HTTP_201_CREATED)
def create_product(product: ProductOut):
    return product

@app4.delete("/products/{id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_product(id: int):
    return None  # 204 returns no body

# ── response_model_exclude / include ─────────────────────────
@app4.get("/products/{id}/summary",
          response_model=Product,
          response_model_exclude={"secret_key", "description"})
def product_summary(id: int):
    return Product(id=id, name="Phone", price=599.0, description="Nice phone", secret_key="xyz")

# ── Different response types ──────────────────────────────────
@app4.get("/html", response_class=HTMLResponse)
def html_page():
    return "<h1>Hello from FastAPI!</h1><p>This is HTML.</p>"

@app4.get("/text", response_class=PlainTextResponse)
def plain_text():
    return "Hello, plain text!"

@app4.get("/redirect")
def redirect_me():
    return RedirectResponse(url="/docs", status_code=302)

# ── Custom JSONResponse ───────────────────────────────────────
@app4.get("/custom-response")
def custom_response():
    return JSONResponse(
        content={"status": "ok", "data": [1, 2, 3]},
        status_code=200,
        headers={"X-Custom-Header": "fastapi"}
    )

# ── Multiple response schemas documented ─────────────────────
@app4.get("/items/{id}", responses={
    200: {"model": ProductOut, "description": "Item found"},
    404: {"description": "Item not found"},
    500: {"description": "Server error"}
})
def get_item(id: int):
    if id > 100:
        return JSONResponse(status_code=404, content={"detail": "Not found"})
    return ProductOut(id=id, name="Widget", price=9.99)

In [ ]:
server4 = run_app(app4, port=8003)

with httpx.Client(base_url="http://127.0.0.1:8003", follow_redirects=False) as c:
    r = c.get("/products/1")
    print("Product (secret_key filtered):", r.json())

    r = c.post("/products", json={"id": 2, "name": "Phone", "price": 599.0})
    print("Created (201):", r.status_code, r.json())

    r = c.delete("/products/1")
    print("Deleted (204):", r.status_code)  # no body

    r = c.get("/custom-response")
    print("Custom header:", r.headers.get("x-custom-header"), r.json())

    r = c.get("/items/999")
    print("Not found:", r.status_code, r.json())

---
## 6. Headers, Cookies & Form Data

In [ ]:
from fastapi import FastAPI, Header, Cookie, Form, Response
from typing import Optional

app5 = FastAPI()

# ── Reading request headers ───────────────────────────────────
@app5.get("/headers")
def read_headers(
    user_agent:    Optional[str] = Header(default=None),
    authorization: Optional[str] = Header(default=None),
    x_request_id:  Optional[str] = Header(default=None)  # X-Request-Id → x_request_id
):
    return {
        "user_agent":    user_agent,
        "authorization": authorization,
        "request_id":    x_request_id
    }

# ── Setting response headers ──────────────────────────────────
@app5.get("/set-headers")
def set_custom_headers(response: Response):
    response.headers["X-Process-Time"] = "0.01s"
    response.headers["X-API-Version"]  = "1.0"
    return {"message": "Headers set"}

# ── Cookies: read ─────────────────────────────────────────────
@app5.get("/profile")
def read_cookie(
    session_token: Optional[str] = Cookie(default=None),
    user_pref:     Optional[str] = Cookie(default=None)
):
    return {"session": session_token, "prefs": user_pref}

# ── Cookies: set & delete ─────────────────────────────────────
@app5.post("/login")
def login(response: Response, username: str = Form(...), password: str = Form(...)):
    # Validate credentials (demo only)
    if username == "admin" and password == "secret":
        response.set_cookie(
            key="session_token",
            value="abc123xyz",
            httponly=True,
            max_age=3600,
            samesite="lax"
        )
        return {"message": "Logged in"}
    return {"error": "Invalid credentials"}

@app5.post("/logout")
def logout(response: Response):
    response.delete_cookie("session_token")
    return {"message": "Logged out"}

# ── Form data ─────────────────────────────────────────────────
@app5.post("/register")
def register(
    username: str  = Form(..., min_length=3),
    email:    str  = Form(...),
    password: str  = Form(..., min_length=8),
    agree:    bool = Form(...)
):
    return {"registered": username, "email": email, "agreed": agree}

In [ ]:
server5 = run_app(app5, port=8004)

with httpx.Client(base_url="http://127.0.0.1:8004") as c:
    r = c.get("/headers", headers={"X-Request-Id": "req-001"})
    print("Headers read:", r.json())

    r = c.get("/set-headers")
    print("Response headers:", dict(r.headers))

    r = c.post("/login", data={"username": "admin", "password": "secret"})
    print("Login:", r.json(), "| Set-Cookie:", r.headers.get("set-cookie"))

    r = c.post("/register", data={
        "username": "alice", "email": "alice@example.com",
        "password": "securepass", "agree": "true"
    })
    print("Register:", r.json())

---
## 7. File Uploads

FastAPI handles single and multiple file uploads with `File` and `UploadFile`.

In [ ]:
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import JSONResponse
from typing import List
import shutil, os

app6 = FastAPI()
UPLOAD_DIR = "/tmp/uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# ── Single file upload ────────────────────────────────────────
@app6.post("/upload")
async def upload_file(file: UploadFile = File(...)):
    allowed_types = {"image/jpeg", "image/png", "application/pdf", "text/plain"}
    if file.content_type not in allowed_types:
        return JSONResponse(status_code=400, content={"error": f"Type '{file.content_type}' not allowed"})

    dest = os.path.join(UPLOAD_DIR, file.filename)
    with open(dest, "wb") as f:
        shutil.copyfileobj(file.file, f)

    return {
        "filename":     file.filename,
        "content_type": file.content_type,
        "size_bytes":   os.path.getsize(dest)
    }

# ── Multiple files ────────────────────────────────────────────
@app6.post("/upload-multiple")
async def upload_multiple(files: List[UploadFile] = File(...)):
    results = []
    for file in files:
        content = await file.read()
        results.append({"filename": file.filename, "size": len(content)})
    return {"uploaded": results}

# ── File + form fields together ───────────────────────────────
@app6.post("/upload-with-meta")
async def upload_with_meta(
    file:        UploadFile = File(...),
    description: str        = Form(...),
    category:    str        = Form(default="general")
):
    content = await file.read()
    return {
        "filename":    file.filename,
        "size":        len(content),
        "description": description,
        "category":    category
    }

# ── Read and process file content ────────────────────────────
@app6.post("/process-text")
async def process_text(file: UploadFile = File(...)):
    if not file.filename.endswith(".txt"):
        return JSONResponse(status_code=400, content={"error": "Only .txt files accepted"})
    content = await file.read()
    text = content.decode("utf-8")
    return {
        "filename":   file.filename,
        "characters": len(text),
        "words":      len(text.split()),
        "lines":      text.count("\n") + 1,
        "preview":    text[:200]
    }

In [ ]:
server6 = run_app(app6, port=8005)

with httpx.Client(base_url="http://127.0.0.1:8005") as c:
    # Upload a text file
    text_content = b"Hello FastAPI!\nThis is a test file.\nLine three."
    r = c.post("/process-text", files={"file": ("test.txt", text_content, "text/plain")})
    print("Process text:", r.json())

    # Upload with metadata
    r = c.post("/upload-with-meta",
               files={"file": ("doc.txt", b"Sample content", "text/plain")},
               data={"description": "My document", "category": "reports"})
    print("Upload+meta:", r.json())

    # Multiple files
    r = c.post("/upload-multiple", files=[
        ("files", ("a.txt", b"file a content", "text/plain")),
        ("files", ("b.txt", b"file b content", "text/plain")),
    ])
    print("Multiple:", r.json())

---
## 8. Dependency Injection

`Depends()` is FastAPI's powerful DI system for reusable logic, shared resources, and layered validation.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Query
from typing import Optional

app7 = FastAPI()

# ── Simple function dependency ────────────────────────────────
def common_pagination(page: int = Query(1, ge=1), limit: int = Query(10, ge=1, le=100)):
    return {"skip": (page - 1) * limit, "limit": limit}

@app7.get("/items")
def list_items(pagination: dict = Depends(common_pagination)):
    return {"items": [], "pagination": pagination}

@app7.get("/users")
def list_users(pagination: dict = Depends(common_pagination)):
    return {"users": [], "pagination": pagination}

# ── Class-based dependency ────────────────────────────────────
class DatabaseSession:
    def __init__(self):
        self.connected = True
        print("  [DB] Session opened")

    def query(self, table: str):
        return [{"id": 1, "table": table}]

    def close(self):
        self.connected = False
        print("  [DB] Session closed")

def get_db():
    db = DatabaseSession()
    try:
        yield db      # Provide to endpoint
    finally:
        db.close()    # Always clean up

@app7.get("/db-items")
def get_db_items(db: DatabaseSession = Depends(get_db)):
    return db.query("items")

# ── Nested dependencies ───────────────────────────────────────
def get_api_key(x_api_key: Optional[str] = Query(default=None)):
    if x_api_key != "secret-key":
        raise HTTPException(status_code=403, detail="Invalid API key")
    return x_api_key

def get_current_user(api_key: str = Depends(get_api_key)):
    return {"user": "alice", "role": "admin", "api_key": api_key}

@app7.get("/protected")
def protected_route(user: dict = Depends(get_current_user)):
    return {"message": f"Hello {user['user']}!", "role": user["role"]}

# ── Router-level dependency (applies to all routes) ───────────
from fastapi import APIRouter

admin_router = APIRouter(prefix="/admin", dependencies=[Depends(get_api_key)])

@admin_router.get("/stats")
def admin_stats():
    return {"total_users": 1000, "revenue": 50000}

@admin_router.get("/logs")
def admin_logs():
    return {"logs": ["event1", "event2"]}

app7.include_router(admin_router)

In [ ]:
server7 = run_app(app7, port=8006)

with httpx.Client(base_url="http://127.0.0.1:8006") as c:
    print(c.get("/items", params={"page": 2, "limit": 5}).json())
    print(c.get("/db-items").json())
    print(c.get("/protected", params={"x_api_key": "secret-key"}).json())
    print(c.get("/protected", params={"x_api_key": "wrong"}).json())
    print(c.get("/admin/stats", params={"x_api_key": "secret-key"}).json())

---
## 9. Authentication & Security

FastAPI provides OAuth2, JWT, HTTP Basic, and API Key auth out of the box.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm, HTTPBearer, HTTPBasic, HTTPBasicCredentials, APIKeyHeader
from jose import JWTError, jwt
from passlib.context import CryptContext
from pydantic import BaseModel
from datetime import datetime, timedelta
from typing import Optional
import secrets

SECRET_KEY = "your-super-secret-key-change-in-production"
ALGORITHM  = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

pwd_context   = CryptContext(schemes=["bcrypt"], deprecated="auto")
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/token")

# Fake user database
FAKE_USERS = {
    "alice": {"username": "alice", "hashed_password": pwd_context.hash("password123"), "role": "admin"},
    "bob":   {"username": "bob",   "hashed_password": pwd_context.hash("bob456"),       "role": "user"},
}

class Token(BaseModel):
    access_token: str
    token_type:   str

class TokenData(BaseModel):
    username: Optional[str] = None

def verify_password(plain: str, hashed: str) -> bool:
    return pwd_context.verify(plain, hashed)

def create_access_token(data: dict, expires_delta: Optional[timedelta] = None) -> str:
    to_encode = data.copy()
    expire = datetime.utcnow() + (expires_delta or timedelta(minutes=15))
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

def get_current_user(token: str = Depends(oauth2_scheme)):
    credentials_exception = HTTPException(
        status_code=status.HTTP_401_UNAUTHORIZED,
        detail="Could not validate credentials",
        headers={"WWW-Authenticate": "Bearer"}
    )
    try:
        payload  = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        username = payload.get("sub")
        if username is None:
            raise credentials_exception
    except JWTError:
        raise credentials_exception

    user = FAKE_USERS.get(username)
    if user is None:
        raise credentials_exception
    return user

def require_admin(user: dict = Depends(get_current_user)):
    if user["role"] != "admin":
        raise HTTPException(status_code=403, detail="Admins only")
    return user

app8 = FastAPI()

@app8.post("/auth/token", response_model=Token)
def login(form: OAuth2PasswordRequestForm = Depends()):
    user = FAKE_USERS.get(form.username)
    if not user or not verify_password(form.password, user["hashed_password"]):
        raise HTTPException(status_code=401, detail="Incorrect username or password")
    token = create_access_token(
        data={"sub": user["username"]},
        expires_delta=timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    )
    return {"access_token": token, "token_type": "bearer"}

@app8.get("/auth/me")
def read_me(user: dict = Depends(get_current_user)):
    return {"username": user["username"], "role": user["role"]}

@app8.get("/admin/dashboard")
def admin_dashboard(user: dict = Depends(require_admin)):
    return {"message": f"Welcome admin {user['username']}!", "stats": {"users": 500}}

In [ ]:
server8 = run_app(app8, port=8007)

with httpx.Client(base_url="http://127.0.0.1:8007") as c:
    # Login to get token
    r = c.post("/auth/token", data={"username": "alice", "password": "password123"})
    token = r.json()["access_token"]
    print("Token obtained:", token[:40], "...")

    # Use token
    headers = {"Authorization": f"Bearer {token}"}
    print("Me:", c.get("/auth/me", headers=headers).json())
    print("Admin:", c.get("/admin/dashboard", headers=headers).json())

    # Bob can't access admin
    r_bob = c.post("/auth/token", data={"username": "bob", "password": "bob456"})
    bob_token = r_bob.json()["access_token"]
    print("Bob admin access:", c.get("/admin/dashboard", headers={"Authorization": f"Bearer {bob_token}"}).json())

In [ ]:
# ── API Key Authentication ─────────────────────────────────────
API_KEY_NAME = "X-API-Key"
VALID_API_KEYS = {"key-abc-123", "key-xyz-789"}

api_key_header = APIKeyHeader(name=API_KEY_NAME, auto_error=True)

def verify_api_key(api_key: str = Depends(api_key_header)):
    if api_key not in VALID_API_KEYS:
        raise HTTPException(status_code=403, detail="Invalid API Key")
    return api_key

app_apikey = FastAPI()

@app_apikey.get("/data")
def get_data(api_key: str = Depends(verify_api_key)):
    return {"data": [1, 2, 3], "key_used": api_key[:8] + "..."}

srv = run_app(app_apikey, port=8008)
with httpx.Client(base_url="http://127.0.0.1:8008") as c:
    print(c.get("/data", headers={"X-API-Key": "key-abc-123"}).json())
    print(c.get("/data", headers={"X-API-Key": "wrong-key"}).json())

---
## 10. Database Integration (SQLAlchemy)

FastAPI works seamlessly with SQLAlchemy for sync and async database access.

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, Boolean, ForeignKey
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, Session, relationship
from fastapi import FastAPI, Depends, HTTPException
from pydantic import BaseModel
from typing import List, Optional

# ── Database setup ─────────────────────────────────────────────
DATABASE_URL = "sqlite:////tmp/fastapi_demo.db"

engine       = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base         = declarative_base()

# ── SQLAlchemy Models ──────────────────────────────────────────
class UserDB(Base):
    __tablename__ = "users"
    id       = Column(Integer, primary_key=True, index=True)
    name     = Column(String, nullable=False)
    email    = Column(String, unique=True, index=True, nullable=False)
    is_active= Column(Boolean, default=True)
    items    = relationship("ItemDB", back_populates="owner")

class ItemDB(Base):
    __tablename__ = "items"
    id       = Column(Integer, primary_key=True, index=True)
    name     = Column(String, nullable=False)
    price    = Column(Float, nullable=False)
    owner_id = Column(Integer, ForeignKey("users.id"))
    owner    = relationship("UserDB", back_populates="items")

Base.metadata.create_all(bind=engine)

# ── Pydantic schemas ───────────────────────────────────────────
class ItemCreate(BaseModel):
    name:  str
    price: float

class ItemOut(BaseModel):
    id:    int
    name:  str
    price: float
    model_config = {"from_attributes": True}

class UserCreate(BaseModel):
    name:  str
    email: str

class UserOut(BaseModel):
    id:       int
    name:     str
    email:    str
    is_active:bool
    items:    List[ItemOut] = []
    model_config = {"from_attributes": True}

# ── Dependency: DB session ─────────────────────────────────────
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

# ── FastAPI App ────────────────────────────────────────────────
app9 = FastAPI()

@app9.post("/users", response_model=UserOut, status_code=201)
def create_user(user: UserCreate, db: Session = Depends(get_db)):
    if db.query(UserDB).filter(UserDB.email == user.email).first():
        raise HTTPException(status_code=400, detail="Email already registered")
    db_user = UserDB(name=user.name, email=user.email)
    db.add(db_user)
    db.commit()
    db.refresh(db_user)
    return db_user

@app9.get("/users", response_model=List[UserOut])
def list_users(skip: int = 0, limit: int = 10, db: Session = Depends(get_db)):
    return db.query(UserDB).offset(skip).limit(limit).all()

@app9.get("/users/{user_id}", response_model=UserOut)
def get_user(user_id: int, db: Session = Depends(get_db)):
    user = db.query(UserDB).filter(UserDB.id == user_id).first()
    if not user:
        raise HTTPException(status_code=404, detail="User not found")
    return user

@app9.post("/users/{user_id}/items", response_model=ItemOut, status_code=201)
def create_item_for_user(user_id: int, item: ItemCreate, db: Session = Depends(get_db)):
    user = db.query(UserDB).filter(UserDB.id == user_id).first()
    if not user:
        raise HTTPException(status_code=404, detail="User not found")
    db_item = ItemDB(**item.model_dump(), owner_id=user_id)
    db.add(db_item)
    db.commit()
    db.refresh(db_item)
    return db_item

@app9.delete("/users/{user_id}", status_code=204)
def delete_user(user_id: int, db: Session = Depends(get_db)):
    user = db.query(UserDB).filter(UserDB.id == user_id).first()
    if not user:
        raise HTTPException(status_code=404, detail="User not found")
    db.delete(user)
    db.commit()

In [ ]:
server9 = run_app(app9, port=8009)

with httpx.Client(base_url="http://127.0.0.1:8009") as c:
    u = c.post("/users", json={"name": "Alice", "email": "alice@db.com"}).json()
    print("Created:", u)

    item = c.post(f"/users/{u['id']}/items", json={"name": "Laptop", "price": 999.99}).json()
    print("Item:", item)

    print("User with items:", c.get(f"/users/{u['id']}").json())
    print("All users:", c.get("/users").json())

---
## 11. Async Endpoints

Use `async def` for I/O-bound tasks (DB, HTTP calls, file I/O) to maximize throughput.

In [3]:
import asyncio
import httpx
from fastapi import FastAPI
from typing import List

app10 = FastAPI()

# ── Basic async endpoint ──────────────────────────────────────
@app10.get("/async-hello")
async def async_hello():
    await asyncio.sleep(0)  # non-blocking yield
    return {"message": "Hello from async!"}

# ── Async external HTTP calls ─────────────────────────────────
@app10.get("/fetch-data")
async def fetch_external_data():
    async with httpx.AsyncClient() as client:
        r = await client.get("https://jsonplaceholder.typicode.com/todos/1")
        return r.json()

# ── Parallel async calls ──────────────────────────────────────
@app10.get("/parallel")
async def parallel_requests():
    async with httpx.AsyncClient() as client:
        tasks = [
            client.get(f"https://jsonplaceholder.typicode.com/todos/{i}")
            for i in range(1, 4)
        ]
        responses = await asyncio.gather(*tasks)
        return [r.json() for r in responses]

# ── Async with simulated DB ───────────────────────────────────
async def fake_async_db_query(user_id: int):
    await asyncio.sleep(0.01)  # simulate DB latency
    return {"id": user_id, "name": "Alice", "email": "alice@example.com"}

@app10.get("/async-user/{user_id}")
async def async_user(user_id: int):
    user = await fake_async_db_query(user_id)
    return user

# ── Async generator streaming response ───────────────────────
from fastapi.responses import StreamingResponse
import json

@app10.get("/stream")
async def stream_data():
    async def generate():
        for i in range(5):
            await asyncio.sleep(0.1)
            yield json.dumps({"chunk": i}) + "\n"
    return StreamingResponse(generate(), media_type="application/x-ndjson")

In [2]:
server10 = run_app(app10, port=8010)

with httpx.Client(base_url="http://127.0.0.1:8010") as c:
    print(c.get("/async-hello").json())
    print(c.get("/fetch-data").json())
    print(c.get("/async-user/42").json())

    # Streaming
    print("\nStreaming chunks:")
    with c.stream("GET", "/stream") as r:
        for line in r.iter_lines():
            if line:
                print(" ", json.loads(line))

NameError: name 'run_app' is not defined

---
## 12. Background Tasks

Run tasks after returning a response (emails, logging, cleanup).

In [ ]:
import asyncio, time
from fastapi import FastAPI, BackgroundTasks, Depends

app11 = FastAPI()
task_log = []  # Simple in-memory log

# ── Background task functions ─────────────────────────────────
def send_email(to: str, subject: str, body: str):
    time.sleep(0.05)  # Simulate sending
    task_log.append({"type": "email", "to": to, "subject": subject})
    print(f"  [BG] Email sent to {to}: {subject}")

def write_audit_log(event: str, user_id: int):
    task_log.append({"type": "audit", "event": event, "user_id": user_id})
    print(f"  [BG] Audit: {event} by user {user_id}")

async def async_cleanup(resource_id: int):
    await asyncio.sleep(0.02)
    task_log.append({"type": "cleanup", "resource_id": resource_id})
    print(f"  [BG] Cleaned up resource {resource_id}")

# ── Endpoints using background tasks ─────────────────────────
@app11.post("/register")
def register_user(username: str, email: str, background_tasks: BackgroundTasks):
    # Add multiple background tasks
    background_tasks.add_task(send_email, email, "Welcome!", f"Hi {username}, welcome!")
    background_tasks.add_task(write_audit_log, "user_registered", 1)
    return {"message": f"User {username} registered. Welcome email queued."}  # Returned immediately

@app11.delete("/resources/{resource_id}")
async def delete_resource(resource_id: int, background_tasks: BackgroundTasks):
    background_tasks.add_task(async_cleanup, resource_id)
    return {"message": f"Resource {resource_id} deleted. Cleanup in progress."}

@app11.get("/task-log")
def get_task_log():
    return {"tasks_run": len(task_log), "log": task_log}

In [ ]:
server11 = run_app(app11, port=8011)

with httpx.Client(base_url="http://127.0.0.1:8011") as c:
    r = c.post("/register", params={"username": "alice", "email": "alice@example.com"})
    print("Register response (immediate):", r.json())

    c.delete("/resources/42")
    time.sleep(0.2)  # Wait for background tasks

    print("Task log:", c.get("/task-log").json())

---
## 13. Middleware

Middleware runs before and after every request — great for logging, timing, auth, etc.

In [ ]:
import time, uuid
from fastapi import FastAPI, Request, Response
from starlette.middleware.base import BaseHTTPMiddleware

app12 = FastAPI()

# ── Timing middleware ─────────────────────────────────────────
@app12.middleware("http")
async def add_process_time(request: Request, call_next):
    start = time.time()
    response = await call_next(request)
    duration = time.time() - start
    response.headers["X-Process-Time"] = f"{duration:.4f}s"
    return response

# ── Request ID middleware ─────────────────────────────────────
@app12.middleware("http")
async def add_request_id(request: Request, call_next):
    request_id = request.headers.get("X-Request-Id", str(uuid.uuid4())[:8])
    response = await call_next(request)
    response.headers["X-Request-Id"] = request_id
    return response

# ── Class-based middleware ─────────────────────────────────────
class LoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        print(f"  [LOG] {request.method} {request.url.path}")
        response = await call_next(request)
        print(f"  [LOG] Response status: {response.status_code}")
        return response

app12.add_middleware(LoggingMiddleware)

# ── Auth middleware ───────────────────────────────────────────
class SimpleAuthMiddleware(BaseHTTPMiddleware):
    EXCLUDED_PATHS = {"/", "/health", "/docs", "/openapi.json"}

    async def dispatch(self, request: Request, call_next):
        if request.url.path not in self.EXCLUDED_PATHS:
            api_key = request.headers.get("X-API-Key")
            if api_key != "valid-key":
                return Response(content="Unauthorized", status_code=401)
        return await call_next(request)

# app12.add_middleware(SimpleAuthMiddleware)  # Uncomment to enable

@app12.get("/")
def root():
    return {"message": "Hello"}

@app12.get("/slow")
async def slow_endpoint():
    await asyncio.sleep(0.05)
    return {"result": "done"}

In [ ]:
server12 = run_app(app12, port=8012)

with httpx.Client(base_url="http://127.0.0.1:8012") as c:
    r = c.get("/")
    print("Process time:", r.headers.get("x-process-time"))
    print("Request ID:", r.headers.get("x-request-id"))

    r = c.get("/slow")
    print("Slow endpoint time:", r.headers.get("x-process-time"))

---
## 14. CORS

Cross-Origin Resource Sharing — essential for browser-based frontends.

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app13 = FastAPI()

# ── CORS configuration ─────────────────────────────────────────
app13.add_middleware(
    CORSMiddleware,
    allow_origins=[          # Specific origins (use ["*"] for all)
        "http://localhost:3000",
        "https://myapp.com",
        "https://www.myapp.com"
    ],
    allow_credentials=True,  # Allow cookies to be sent cross-origin
    allow_methods=["GET", "POST", "PUT", "DELETE", "PATCH"],
    allow_headers=["*"],     # Allow all headers
    expose_headers=["X-Process-Time", "X-Request-Id"],
    max_age=600              # Cache preflight response for 10 minutes
)

@app13.get("/api/data")
def get_data():
    return {"data": "This can be fetched from allowed origins"}

print("CORS configured for:")
print("  Origins : http://localhost:3000, https://myapp.com")
print("  Methods : GET, POST, PUT, DELETE, PATCH")
print("  Headers : all")
print("  Creds   : allowed")

---
## 15. WebSockets

FastAPI supports full WebSocket communication for real-time features.

In [ ]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from typing import List

app14 = FastAPI()

# ── Simple echo WebSocket ─────────────────────────────────────
@app14.websocket("/ws/echo")
async def ws_echo(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            data = await websocket.receive_text()
            await websocket.send_text(f"Echo: {data}")
    except WebSocketDisconnect:
        print("  [WS] Client disconnected from echo")

# ── Connection Manager for broadcast ─────────────────────────
class ConnectionManager:
    def __init__(self):
        self.active_connections: List[WebSocket] = []

    async def connect(self, ws: WebSocket):
        await ws.accept()
        self.active_connections.append(ws)

    def disconnect(self, ws: WebSocket):
        self.active_connections.remove(ws)

    async def broadcast(self, message: str):
        for conn in self.active_connections:
            await conn.send_text(message)

    async def send_personal(self, message: str, ws: WebSocket):
        await ws.send_text(message)

manager = ConnectionManager()

@app14.websocket("/ws/chat/{username}")
async def ws_chat(websocket: WebSocket, username: str):
    await manager.connect(websocket)
    await manager.broadcast(f"{username} joined the chat. ({len(manager.active_connections)} online)")
    try:
        while True:
            msg = await websocket.receive_text()
            await manager.broadcast(f"{username}: {msg}")
    except WebSocketDisconnect:
        manager.disconnect(websocket)
        await manager.broadcast(f"{username} left the chat.")

# ── JSON WebSocket ────────────────────────────────────────────
@app14.websocket("/ws/json")
async def ws_json(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            data = await websocket.receive_json()
            response = {"received": data, "processed": True, "count": len(data)}
            await websocket.send_json(response)
    except WebSocketDisconnect:
        pass

In [ ]:
server14 = run_app(app14, port=8013)

# Test WebSocket
async def test_websocket():
    async with httpx.AsyncClient(base_url="http://127.0.0.1:8013") as client:
        async with client.stream("GET", "/ws/echo",
                                  headers={"connection": "upgrade", "upgrade": "websocket",
                                           "sec-websocket-key": "dGhlIHNhbXBsZSBub25jZQ==",
                                           "sec-websocket-version": "13"}) as r:
            print("WS handshake status:", r.status_code)

import asyncio
asyncio.run(test_websocket())

print("\nWebSocket endpoints defined:")
print("  ws://localhost:8013/ws/echo       — Echo server")
print("  ws://localhost:8013/ws/chat/{username} — Multi-user chat")
print("  ws://localhost:8013/ws/json       — JSON message exchange")

---
## 16. Error Handling & Custom Exceptions

Handle errors gracefully with custom exception classes and global handlers.

In [ ]:
from fastapi import FastAPI, HTTPException, Request, status
from fastapi.responses import JSONResponse
from fastapi.exception_handlers import http_exception_handler
from fastapi.exceptions import RequestValidationError
from pydantic import BaseModel

app15 = FastAPI()

# ── Custom exception classes ──────────────────────────────────
class AppException(Exception):
    def __init__(self, status_code: int, error_code: str, message: str, details: dict = None):
        self.status_code = status_code
        self.error_code  = error_code
        self.message     = message
        self.details     = details or {}

class NotFoundException(AppException):
    def __init__(self, resource: str, id: int):
        super().__init__(404, "NOT_FOUND", f"{resource} with id={id} not found", {"resource": resource, "id": id})

class RateLimitException(AppException):
    def __init__(self, limit: int):
        super().__init__(429, "RATE_LIMIT", f"Rate limit of {limit} req/min exceeded", {"limit": limit})

class BusinessRuleException(AppException):
    def __init__(self, rule: str):
        super().__init__(422, "BUSINESS_RULE", f"Business rule violated: {rule}")

# ── Register custom exception handler ────────────────────────
@app15.exception_handler(AppException)
async def app_exception_handler(request: Request, exc: AppException):
    return JSONResponse(
        status_code=exc.status_code,
        content={"error": exc.error_code, "message": exc.message, "details": exc.details}
    )

# ── Override default validation error handler ─────────────────
@app15.exception_handler(RequestValidationError)
async def validation_error_handler(request: Request, exc: RequestValidationError):
    errors = []
    for err in exc.errors():
        errors.append({"field": ".".join(str(x) for x in err["loc"]), "message": err["msg"], "type": err["type"]})
    return JSONResponse(status_code=422, content={"error": "VALIDATION_ERROR", "fields": errors})

# ── Override default 404 handler ──────────────────────────────
@app15.exception_handler(404)
async def custom_404(request: Request, exc: HTTPException):
    return JSONResponse(status_code=404, content={"error": "ROUTE_NOT_FOUND", "path": str(request.url.path)})

# ── Global catch-all ──────────────────────────────────────────
@app15.exception_handler(Exception)
async def global_exception_handler(request: Request, exc: Exception):
    return JSONResponse(status_code=500, content={"error": "INTERNAL_ERROR", "message": "An unexpected error occurred"})

# ── Endpoints using exceptions ────────────────────────────────
@app15.get("/users/{user_id}")
def get_user(user_id: int):
    if user_id > 100:
        raise NotFoundException("User", user_id)
    return {"id": user_id, "name": "Alice"}

@app15.post("/transfer")
def transfer(amount: float, balance: float):
    if amount > balance:
        raise BusinessRuleException("Transfer amount exceeds balance")
    return {"transferred": amount}

class Item(BaseModel):
    name: str
    price: float

@app15.post("/items")
def create_item(item: Item):
    return item

In [ ]:
server15 = run_app(app15, port=8014)

with httpx.Client(base_url="http://127.0.0.1:8014") as c:
    print("Valid user:",  c.get("/users/5").json())
    print("Not found:",   c.get("/users/999").json())
    print("Business rule:", c.post("/transfer", params={"amount": 500, "balance": 100}).json())
    print("Validation error:", c.post("/items", json={"name": "x", "price": "not-a-number"}).json())
    print("Route not found:",  c.get("/nonexistent").json())

---
## 17. APIRouter & App Structure

Organize large apps into separate modules using `APIRouter`.

In [ ]:
from fastapi import FastAPI, APIRouter, Depends, HTTPException
from pydantic import BaseModel
from typing import List

# ── Reusable dependency ───────────────────────────────────────
def get_current_user():
    return {"id": 1, "username": "alice", "role": "admin"}

# ── users router (would live in routers/users.py) ─────────────
users_router = APIRouter(
    prefix="/users",
    tags=["Users"],
    dependencies=[Depends(get_current_user)]
)

class UserOut(BaseModel):
    id: int
    name: str

@users_router.get("/", response_model=List[UserOut])
def list_users():
    return [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]

@users_router.get("/{user_id}", response_model=UserOut)
def get_user(user_id: int):
    return {"id": user_id, "name": "Alice"}

# ── products router (would live in routers/products.py) ───────
products_router = APIRouter(prefix="/products", tags=["Products"])

class ProductOut(BaseModel):
    id:    int
    name:  str
    price: float

@products_router.get("/", response_model=List[ProductOut])
def list_products():
    return [{"id": 1, "name": "Laptop", "price": 999.0}]

# ── health router ──────────────────────────────────────────────
health_router = APIRouter(tags=["Health"])

@health_router.get("/health")
def health_check():
    return {"status": "healthy", "version": "1.0.0"}

# ── Main app assembles all routers ────────────────────────────
app16 = FastAPI(title="Structured App", version="1.0.0")

app16.include_router(health_router)
app16.include_router(users_router)
app16.include_router(products_router, prefix="/api/v1")  # version prefix

@app16.get("/")
def root():
    return {"message": "API is running", "docs": "/docs"}

In [ ]:
server16 = run_app(app16, port=8015)

with httpx.Client(base_url="http://127.0.0.1:8015") as c:
    print(c.get("/health").json())
    print(c.get("/users/").json())
    print(c.get("/users/1").json())
    print(c.get("/api/v1/products/").json())

print("\nAll routes:")
for route in app16.routes:
    if hasattr(route, "methods"):
        print(f"  {list(route.methods)} {route.path}")

---
## 18. Testing with TestClient

FastAPI's `TestClient` (based on `httpx`) makes testing clean and fast — no server needed.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import List

# ── App to test ────────────────────────────────────────────────
app_test = FastAPI()

fake_db = {}

class Item(BaseModel):
    name:  str
    price: float

class ItemOut(Item):
    id: int

@app_test.get("/items", response_model=List[ItemOut])
def list_items():
    return list(fake_db.values())

@app_test.post("/items", response_model=ItemOut, status_code=201)
def create_item(item: Item):
    item_id = len(fake_db) + 1
    new_item = ItemOut(id=item_id, **item.model_dump())
    fake_db[item_id] = new_item
    return new_item

@app_test.get("/items/{item_id}", response_model=ItemOut)
def get_item(item_id: int):
    if item_id not in fake_db:
        raise HTTPException(status_code=404, detail="Item not found")
    return fake_db[item_id]

# ── Test suite ─────────────────────────────────────────────────
client = TestClient(app_test)

def test_create_item():
    r = client.post("/items", json={"name": "Widget", "price": 9.99})
    assert r.status_code == 201
    assert r.json()["name"] == "Widget"
    assert r.json()["id"] == 1
    print("✅ test_create_item passed")

def test_list_items():
    r = client.get("/items")
    assert r.status_code == 200
    assert isinstance(r.json(), list)
    print("✅ test_list_items passed")

def test_get_item_not_found():
    r = client.get("/items/9999")
    assert r.status_code == 404
    assert r.json()["detail"] == "Item not found"
    print("✅ test_get_item_not_found passed")

def test_validation_error():
    r = client.post("/items", json={"name": "x", "price": "not-a-float"})
    assert r.status_code == 422
    print("✅ test_validation_error passed")

def test_headers_and_auth():
    r = client.get("/items", headers={"X-Custom-Header": "myvalue"})
    assert r.status_code == 200
    print("✅ test_headers_and_auth passed")

# Run all tests
fake_db.clear()
test_create_item()
test_list_items()
test_get_item_not_found()
test_validation_error()
test_headers_and_auth()
print("\n🎉 All tests passed!")

In [ ]:
# ── Dependency override for testing ───────────────────────────
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

app_dep_test = FastAPI()

def get_db():
    return {"connected": True, "type": "production"}

@app_dep_test.get("/info")
def info(db=Depends(get_db)):
    return {"db_connected": db["connected"], "db_type": db["type"]}

# Override the DB dependency for tests
def get_test_db():
    return {"connected": True, "type": "test-sqlite-in-memory"}

test_client = TestClient(app_dep_test)
app_dep_test.dependency_overrides[get_db] = get_test_db

r = test_client.get("/info")
print(r.json())  # db_type = test-sqlite-in-memory
assert r.json()["db_type"] == "test-sqlite-in-memory"
print("✅ Dependency override works!")

---
## 19. Lifespan Events (Startup/Shutdown)

Run initialization and cleanup code when the application starts and stops.

In [ ]:
from contextlib import asynccontextmanager
from fastapi import FastAPI
import asyncio

# Shared app state
app_state = {}

# ── Modern lifespan approach (recommended) ────────────────────
@asynccontextmanager
async def lifespan(app: FastAPI):
    # ── STARTUP ───────────────────────────────────────────────
    print("🚀 Starting up...")

    # Initialize DB connection pool
    app_state["db_pool"] = {"pool": "connected", "max_connections": 10}
    print("  ✅ DB pool initialized")

    # Load ML model
    await asyncio.sleep(0.01)  # simulate model loading
    app_state["model"] = {"name": "sentiment-v1", "loaded": True}
    print("  ✅ ML model loaded")

    # Load config/cache
    app_state["config"] = {"env": "production", "debug": False}
    print("  ✅ Config loaded")

    yield  # ← App runs here

    # ── SHUTDOWN ──────────────────────────────────────────────
    print("🛑 Shutting down...")
    app_state["db_pool"] = None
    app_state["model"]   = None
    print("  ✅ Resources released")

app17 = FastAPI(lifespan=lifespan)

@app17.get("/status")
def status():
    return {
        "db":    app_state.get("db_pool"),
        "model": app_state.get("model"),
        "config":app_state.get("config")
    }

@app17.post("/predict")
def predict(text: str):
    model = app_state.get("model")
    if not model:
        return {"error": "Model not loaded"}
    return {"text": text, "sentiment": "positive", "model": model["name"]}

In [ ]:
server17 = run_app(app17, port=8016)

with httpx.Client(base_url="http://127.0.0.1:8016") as c:
    print("Status:", c.get("/status").json())
    print("Predict:", c.post("/predict", params={"text": "This is great!"}).json())

---
## 20. Caching & Rate Limiting

Cache expensive responses and throttle requests to protect your API.

In [ ]:
import time, functools, hashlib
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from typing import Any

app18 = FastAPI()

# ── In-memory TTL cache ───────────────────────────────────────
class TTLCache:
    def __init__(self):
        self._store: dict = {}

    def get(self, key: str) -> Any:
        if key in self._store:
            value, expires_at = self._store[key]
            if time.time() < expires_at:
                return value
            del self._store[key]
        return None

    def set(self, key: str, value: Any, ttl: int = 60):
        self._store[key] = (value, time.time() + ttl)

    def clear(self):
        self._store.clear()

cache = TTLCache()

def cached(ttl: int = 60):
    """Decorator to cache endpoint responses."""
    def decorator(func):
        @functools.wraps(func)
        async def wrapper(request: Request, *args, **kwargs):
            key = hashlib.md5(str(request.url).encode()).hexdigest()
            cached_val = cache.get(key)
            if cached_val:
                return JSONResponse(content=cached_val, headers={"X-Cache": "HIT"})
            result = await func(request, *args, **kwargs) if asyncio.iscoroutinefunction(func) else func(request, *args, **kwargs)
            cache.set(key, result, ttl)
            return JSONResponse(content=result, headers={"X-Cache": "MISS"})
        return wrapper
    return decorator

@app18.get("/expensive-data")
@cached(ttl=30)
def expensive_data(request: Request):
    time.sleep(0.1)  # Simulate expensive computation
    return {"data": "computed result", "timestamp": time.time()}

# ── In-memory Rate Limiter ─────────────────────────────────────
from collections import defaultdict

class RateLimiter:
    def __init__(self, max_calls: int, period: int):
        self.max_calls = max_calls
        self.period    = period
        self._calls    = defaultdict(list)

    def is_allowed(self, key: str) -> bool:
        now = time.time()
        self._calls[key] = [t for t in self._calls[key] if t > now - self.period]
        if len(self._calls[key]) < self.max_calls:
            self._calls[key].append(now)
            return True
        return False

rate_limiter = RateLimiter(max_calls=5, period=60)  # 5 requests per minute

@app18.get("/limited")
def limited_endpoint(request: Request):
    client_ip = request.client.host
    if not rate_limiter.is_allowed(client_ip):
        return JSONResponse(
            status_code=429,
            content={"error": "Rate limit exceeded. Try again later."},
            headers={"Retry-After": "60"}
        )
    return {"message": "Request allowed", "remaining": rate_limiter.max_calls - len(rate_limiter._calls[client_ip])}

@app18.get("/cache/clear")
def clear_cache():
    cache.clear()
    return {"message": "Cache cleared"}

In [ ]:
server18 = run_app(app18, port=8017)

with httpx.Client(base_url="http://127.0.0.1:8017") as c:
    t0 = time.time()
    r1 = c.get("/expensive-data")
    print(f"First call  ({time.time()-t0:.3f}s) X-Cache={r1.headers.get('x-cache')}: {r1.json()['data']}")

    t0 = time.time()
    r2 = c.get("/expensive-data")
    print(f"Second call ({time.time()-t0:.3f}s) X-Cache={r2.headers.get('x-cache')}: {r2.json()['data']}")

    print("\nRate limit test:")
    for i in range(7):
        r = c.get("/limited")
        print(f"  Call {i+1}: {r.status_code} — {r.json()}")

---
## 21. Deployment & Configuration

Best practices for configuring and deploying FastAPI to production.

In [ ]:
from pydantic_settings import BaseSettings, SettingsConfigDict
from functools import lru_cache

# ── Settings via pydantic-settings (reads .env automatically) ─
class Settings(BaseSettings):
    # App
    app_name:    str  = "My FastAPI App"
    environment: str  = "development"   # development | staging | production
    debug:       bool = False
    version:     str  = "1.0.0"

    # Server
    host: str = "0.0.0.0"
    port: int = 8000

    # Database
    database_url:     str = "sqlite:///./app.db"
    db_pool_size:     int = 5
    db_max_overflow:  int = 10

    # Security
    secret_key:   str = "change-me-in-production"
    jwt_algorithm:str = "HS256"
    access_token_expire_minutes: int = 30

    # CORS
    allowed_origins: list = ["http://localhost:3000"]

    # Redis (optional)
    redis_url: str = "redis://localhost:6379"

    model_config = SettingsConfigDict(
        env_file=".env",           # Load from .env file
        env_file_encoding="utf-8",
        case_sensitive=False
    )

@lru_cache()  # Cache settings to avoid re-reading .env on every call
def get_settings():
    return Settings()

settings = get_settings()
print(f"App: {settings.app_name}")
print(f"Env: {settings.environment}")
print(f"DB:  {settings.database_url}")

In [ ]:
# ── Running with Uvicorn ──────────────────────────────────────

uvicorn_dev_command = """
# Development (auto-reload)
uvicorn main:app --reload --host 0.0.0.0 --port 8000
"""

uvicorn_prod_command = """
# Production (multiple workers)
uvicorn main:app --workers 4 --host 0.0.0.0 --port 8000 --no-access-log
"""

gunicorn_command = """
# Production with Gunicorn + Uvicorn workers
gunicorn main:app -w 4 -k uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000
"""

print("Dev command:", uvicorn_dev_command)
print("Prod command:", uvicorn_prod_command)
print("Gunicorn:", gunicorn_command)

In [ ]:
# ── Dockerfile ────────────────────────────────────────────────
dockerfile = '''
FROM python:3.11-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy app code
COPY . .

# Create non-root user
RUN adduser --disabled-password --gecos '' appuser
USER appuser

EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=5s --start-period=10s \\
  CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]
'''

with open("/tmp/Dockerfile.example", "w") as f:
    f.write(dockerfile)

print("Example Dockerfile:")
print(dockerfile)

In [ ]:
# ── Production-ready app pattern ──────────────────────────────
from contextlib import asynccontextmanager
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.middleware.trustedhost import TrustedHostMiddleware
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@asynccontextmanager
async def lifespan(app: FastAPI):
    logger.info("Starting up application...")
    # Init DB, cache, models here
    yield
    logger.info("Shutting down application...")

def create_app() -> FastAPI:
    settings = get_settings()

    app = FastAPI(
        title=settings.app_name,
        version=settings.version,
        docs_url="/docs" if settings.debug else None,   # Hide docs in prod
        redoc_url="/redoc" if settings.debug else None,
        lifespan=lifespan
    )

    # Middleware
    app.add_middleware(CORSMiddleware,
        allow_origins=settings.allowed_origins,
        allow_credentials=True,
        allow_methods=["*"],
        allow_headers=["*"]
    )
    app.add_middleware(TrustedHostMiddleware, allowed_hosts=["*"])

    # Routes
    @app.get("/health")
    def health():
        return {"status": "ok", "version": settings.version, "env": settings.environment}

    return app

prod_app = create_app()
print("Production app created:", prod_app.title)

---
## 22. Async SQLAlchemy

Use `asyncpg` / `aiosqlite` with SQLAlchemy's async session for fully non-blocking DB access.

In [ ]:
from sqlalchemy.ext.asyncio import create_async_engine, AsyncSession, async_sessionmaker
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from sqlalchemy import select, String, Integer
from fastapi import FastAPI, Depends
from pydantic import BaseModel
from typing import AsyncGenerator, List

# ── Async engine (aiosqlite for SQLite, asyncpg for Postgres) ─
ASYNC_DB_URL = "sqlite+aiosqlite:////tmp/async_demo.db"

async_engine = create_async_engine(ASYNC_DB_URL, echo=False)
AsyncSessionLocal = async_sessionmaker(async_engine, expire_on_commit=False)

class Base(DeclarativeBase):
    pass

# ── Mapped column style (SQLAlchemy 2.0+) ─────────────────────
class NoteDB(Base):
    __tablename__ = "notes"
    id:      Mapped[int]  = mapped_column(Integer, primary_key=True, index=True)
    title:   Mapped[str]  = mapped_column(String(100))
    content: Mapped[str]  = mapped_column(String(1000))

# ── Async DB session dependency ────────────────────────────────
async def get_async_db() -> AsyncGenerator[AsyncSession, None]:
    async with AsyncSessionLocal() as session:
        yield session

# ── Pydantic schemas ───────────────────────────────────────────
class NoteCreate(BaseModel):
    title:   str
    content: str

class NoteOut(NoteCreate):
    id: int
    model_config = {"from_attributes": True}

# ── App + lifespan creates tables ──────────────────────────────
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI):
    async with async_engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)
    yield

app22 = FastAPI(lifespan=lifespan)

@app22.post("/notes", response_model=NoteOut, status_code=201)
async def create_note(note: NoteCreate, db: AsyncSession = Depends(get_async_db)):
    db_note = NoteDB(**note.model_dump())
    db.add(db_note)
    await db.commit()
    await db.refresh(db_note)
    return db_note

@app22.get("/notes", response_model=List[NoteOut])
async def list_notes(db: AsyncSession = Depends(get_async_db)):
    result = await db.execute(select(NoteDB))
    return result.scalars().all()

@app22.get("/notes/{note_id}", response_model=NoteOut)
async def get_note(note_id: int, db: AsyncSession = Depends(get_async_db)):
    result = await db.execute(select(NoteDB).where(NoteDB.id == note_id))
    note = result.scalar_one_or_none()
    if not note:
        from fastapi import HTTPException
        raise HTTPException(status_code=404, detail="Note not found")
    return note

In [ ]:
import httpx, time

server22 = run_app(app22, port=8018)

with httpx.Client(base_url="http://127.0.0.1:8018") as c:
    n1 = c.post("/notes", json={"title": "Hello", "content": "Async SQLAlchemy rocks"}).json()
    n2 = c.post("/notes", json={"title": "FastAPI", "content": "Async all the way"}).json()
    print("Created:", n1)
    print("All notes:", c.get("/notes").json())
    print("Get one:", c.get(f"/notes/{n1['id']}").json())

---
## 23. Pagination Patterns

Three real-world pagination strategies: offset, cursor-based, and keyset.

In [ ]:
from fastapi import FastAPI, Query, HTTPException
from pydantic import BaseModel
from typing import List, Optional, Generic, TypeVar
import base64, json, math

app23 = FastAPI()

# Fake dataset
ITEMS = [{"id": i, "name": f"Item {i}", "price": round(i * 4.99, 2)} for i in range(1, 101)]

T = TypeVar("T")

# ── 1. Offset / Page-based pagination ─────────────────────────
class PagedResponse(BaseModel, Generic[T]):
    items:       list
    total:       int
    page:        int
    page_size:   int
    total_pages: int
    has_next:    bool
    has_prev:    bool

@app23.get("/offset", summary="Offset / page-based pagination")
def offset_pagination(
    page:      int = Query(1,  ge=1),
    page_size: int = Query(10, ge=1, le=50)
):
    total       = len(ITEMS)
    total_pages = math.ceil(total / page_size)
    start       = (page - 1) * page_size
    end         = start + page_size
    return PagedResponse(
        items=ITEMS[start:end], total=total,
        page=page, page_size=page_size, total_pages=total_pages,
        has_next=page < total_pages, has_prev=page > 1
    )

# ── 2. Cursor-based pagination (opaque cursor = base64 offset) ─
class CursorResponse(BaseModel):
    items:       list
    next_cursor: Optional[str]
    prev_cursor: Optional[str]
    has_more:    bool

def encode_cursor(offset: int) -> str:
    return base64.b64encode(json.dumps({"offset": offset}).encode()).decode()

def decode_cursor(cursor: str) -> int:
    return json.loads(base64.b64decode(cursor.encode()).decode())["offset"]

@app23.get("/cursor", summary="Cursor-based pagination")
def cursor_pagination(
    cursor: Optional[str] = Query(None, description="Opaque pagination cursor"),
    limit:  int           = Query(10, ge=1, le=50)
):
    offset = decode_cursor(cursor) if cursor else 0
    page   = ITEMS[offset: offset + limit]
    has_more = (offset + limit) < len(ITEMS)
    return CursorResponse(
        items=page,
        next_cursor=encode_cursor(offset + limit) if has_more else None,
        prev_cursor=encode_cursor(max(0, offset - limit)) if offset > 0 else None,
        has_more=has_more
    )

# ── 3. Keyset pagination (seek method — most efficient for DB) ─
@app23.get("/keyset", summary="Keyset / seek pagination")
def keyset_pagination(
    after_id: Optional[int] = Query(None, description="Return items with id > after_id"),
    limit:    int           = Query(10, ge=1, le=50)
):
    filtered = [i for i in ITEMS if i["id"] > (after_id or 0)]
    page     = filtered[:limit]
    return {
        "items":    page,
        "count":    len(page),
        "has_more": len(filtered) > limit,
        "next_after_id": page[-1]["id"] if page else None
    }

In [ ]:
server23 = run_app(app23, port=8019)

with httpx.Client(base_url="http://127.0.0.1:8019") as c:
    r = c.get("/offset", params={"page": 2, "page_size": 5}).json()
    print(f"Offset  — page {r['page']}/{r['total_pages']}, items: {[i['id'] for i in r['items']]}")

    r1 = c.get("/cursor", params={"limit": 5}).json()
    print(f"Cursor  — first page ids: {[i['id'] for i in r1['items']]}, next_cursor: {r1['next_cursor'][:20]}...")
    r2 = c.get("/cursor", params={"cursor": r1["next_cursor"], "limit": 5}).json()
    print(f"Cursor  — second page ids: {[i['id'] for i in r2['items']]}")

    r = c.get("/keyset", params={"after_id": 10, "limit": 5}).json()
    print(f"Keyset  — ids after 10: {[i['id'] for i in r['items']]}, has_more: {r['has_more']}")

---
## 24. Server-Sent Events (SSE)

SSE streams real-time events from server to browser over a plain HTTP connection — great for dashboards, notifications, and LLM token streaming.

In [ ]:
import asyncio, json, time
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from typing import AsyncGenerator

app24 = FastAPI()

# ── SSE helper: format event data ─────────────────────────────
def sse_format(data: dict | str, event: str = None, id: str = None, retry: int = None) -> str:
    lines = []
    if id:    lines.append(f"id: {id}")
    if event: lines.append(f"event: {event}")
    if retry: lines.append(f"retry: {retry}")
    payload = json.dumps(data) if isinstance(data, dict) else data
    lines.append(f"data: {payload}")
    return "\n".join(lines) + "\n\n"

# ── Simple counter stream ──────────────────────────────────────
@app24.get("/sse/counter")
async def sse_counter(request: Request):
    async def generate() -> AsyncGenerator[str, None]:
        for i in range(1, 6):
            if await request.is_disconnected():
                break
            yield sse_format({"count": i, "timestamp": time.time()}, event="tick", id=str(i))
            await asyncio.sleep(0.2)
        yield sse_format({"message": "Stream complete"}, event="done")

    return StreamingResponse(
        generate(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"}
    )

# ── Live log stream ────────────────────────────────────────────
@app24.get("/sse/logs")
async def sse_logs(request: Request):
    import random
    levels = ["INFO", "INFO", "INFO", "WARNING", "ERROR"]
    messages = ["Request processed", "Cache hit", "DB query ok", "Slow query detected", "Connection timeout"]

    async def generate():
        for i in range(8):
            if await request.is_disconnected(): break
            level = random.choice(levels)
            yield sse_format({"level": level, "msg": random.choice(messages), "seq": i}, event="log")
            await asyncio.sleep(0.15)

    return StreamingResponse(generate(), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache"})

# ── LLM-style token stream ────────────────────────────────────
@app24.get("/sse/llm")
async def sse_llm_stream(prompt: str = "Hello"):
    tokens = f"You asked: '{prompt}'. Here is a streamed response token by token.".split()

    async def generate():
        for token in tokens:
            yield sse_format({"token": token + " "}, event="token")
            await asyncio.sleep(0.05)
        yield sse_format({"done": True}, event="end")

    return StreamingResponse(generate(), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache"})

In [ ]:
server24 = run_app(app24, port=8020)

print("=== SSE Counter ===")
with httpx.Client(base_url="http://127.0.0.1:8020", timeout=10) as c:
    with c.stream("GET", "/sse/counter") as r:
        for line in r.iter_lines():
            if line.startswith("data:"):
                print(" ", json.loads(line[5:]))

print("\n=== LLM Token Stream ===")
with httpx.Client(base_url="http://127.0.0.1:8020", timeout=10) as c:
    with c.stream("GET", "/sse/llm", params={"prompt": "What is FastAPI?"}) as r:
        tokens = []
        for line in r.iter_lines():
            if line.startswith("data:"):
                d = json.loads(line[5:])
                if "token" in d:
                    tokens.append(d["token"])
        print(" ", "".join(tokens))

---
## 25. Static Files & Jinja2 Templates

Serve HTML pages, CSS, JS, and images directly from FastAPI.

In [ ]:
# pip install jinja2
import os
from fastapi import FastAPI, Request
from fastapi.staticfiles import StaticFiles
from fastapi.templating import Jinja2Templates
from fastapi.responses import HTMLResponse

# ── Setup directories ──────────────────────────────────────────
os.makedirs("/tmp/static/css", exist_ok=True)
os.makedirs("/tmp/templates", exist_ok=True)

with open("/tmp/static/css/style.css", "w") as f:
    f.write("body { font-family: Arial; background: #f0f2f5; } h1 { color: #333; }")

# ── Base template with inheritance ────────────────────────────
with open("/tmp/templates/base.html", "w") as f:
    f.write("""
<!DOCTYPE html><html><head>
  <title>{% block title %}My App{% endblock %}</title>
  <link rel="stylesheet" href="/static/css/style.css">
</head><body>
  <nav><strong>MyApp</strong> | <a href="/">Home</a> | <a href="/about">About</a></nav>
  <main>{% block content %}{% endblock %}</main>
  <footer><small>Powered by FastAPI</small></footer>
</body></html>
""")

with open("/tmp/templates/index.html", "w") as f:
    f.write("""
{% extends "base.html" %}
{% block title %}Home — {{ app_name }}{% endblock %}
{% block content %}
  <h1>Welcome, {{ user }}!</h1>
  <p>You have {{ count }} notifications.</p>
  <ul>{% for item in items %}<li>{{ item }}</li>{% endfor %}</ul>
{% endblock %}
""")

with open("/tmp/templates/profile.html", "w") as f:
    f.write("""
{% extends "base.html" %}
{% block title %}Profile — {{ user.name }}{% endblock %}
{% block content %}
  <h2>{{ user.name }}</h2>
  <p>Email: {{ user.email }}</p>
  <p>Role: <span style="color:green">{{ user.role }}</span></p>
{% endblock %}
""")

# ── FastAPI app with templates ────────────────────────────────
app25 = FastAPI()
app25.mount("/static", StaticFiles(directory="/tmp/static"), name="static")
templates = Jinja2Templates(directory="/tmp/templates")

@app25.get("/", response_class=HTMLResponse)
def home(request: Request):
    return templates.TemplateResponse("index.html", {
        "request":  request,
        "app_name": "FastAPI App",
        "user":     "Alice",
        "count":    5,
        "items":    ["Task 1", "Task 2", "Task 3"]
    })

@app25.get("/profile/{username}", response_class=HTMLResponse)
def profile(request: Request, username: str):
    return templates.TemplateResponse("profile.html", {
        "request": request,
        "user":    {"name": username.title(), "email": f"{username}@example.com", "role": "admin"}
    })

@app25.get("/api/ping")  # Mix HTML + API routes
def ping():
    return {"pong": True}

In [ ]:
server25 = run_app(app25, port=8021)

with httpx.Client(base_url="http://127.0.0.1:8021") as c:
    r = c.get("/")
    print("Home status:", r.status_code, "| Content-Type:", r.headers["content-type"])
    print("Has 'Alice':", "Alice" in r.text)

    r = c.get("/profile/alice")
    print("Profile status:", r.status_code, "| Has 'Alice':", "Alice" in r.text)

    r = c.get("/static/css/style.css")
    print("Static CSS:", r.status_code, r.text[:50])

---
## 26. OpenAPI Customization

Customize Swagger UI, add metadata, tags, examples, and security schemes.

In [ ]:
from fastapi import FastAPI, Body
from fastapi.openapi.utils import get_openapi
from fastapi.openapi.docs import get_swagger_ui_html, get_redoc_html
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field
from typing import Annotated

# ── Tag groups for organized docs ─────────────────────────────
tags_metadata = [
    {"name": "users",    "description": "User management operations."},
    {"name": "products", "description": "Product catalog. Requires **admin** role."},
    {"name": "health",   "description": "Health check and system status."},
]

app26 = FastAPI(
    title="My Production API",
    description="""
## My Production API
A fully documented FastAPI application.

### Features
- ✅ JWT Authentication
- ✅ Role-based access control
- ✅ Full CRUD operations
""",
    version="2.1.0",
    terms_of_service="https://example.com/terms",
    contact={"name": "API Support", "url": "https://example.com/support", "email": "support@example.com"},
    license_info={"name": "MIT", "url": "https://opensource.org/licenses/MIT"},
    openapi_tags=tags_metadata,
    docs_url=None,   # Disable default docs (we'll serve custom)
    redoc_url=None
)

# ── Custom Swagger UI with custom CDN / settings ───────────────
@app26.get("/docs", include_in_schema=False)
async def custom_swagger():
    return get_swagger_ui_html(
        openapi_url="/openapi.json",
        title="My API — Swagger UI",
        swagger_ui_parameters={
            "defaultModelsExpandDepth": -1,   # hide schema section
            "docExpansion": "list",            # list | full | none
            "filter": True,                   # show search box
            "syntaxHighlight.theme": "monokai"
        }
    )

@app26.get("/redoc", include_in_schema=False)
async def custom_redoc():
    return get_redoc_html(openapi_url="/openapi.json", title="My API — ReDoc")

# ── Endpoint with rich OpenAPI metadata ───────────────────────
class UserCreate(BaseModel):
    name:  str  = Field(..., examples=["Alice Smith"])
    email: str  = Field(..., examples=["alice@example.com"])
    age:   int  = Field(..., ge=0, le=120, examples=[30])

    model_config = {
        "json_schema_extra": {
            "examples": [{"name": "Alice Smith", "email": "alice@example.com", "age": 30}]
        }
    }

@app26.post(
    "/users",
    tags=["users"],
    summary="Create a new user",
    description="Creates a new user account. Requires a unique email address.",
    response_description="The created user object with assigned ID",
    status_code=201,
    operation_id="create_user_v2"
)
def create_user(user: UserCreate):
    return {"id": 1, **user.model_dump()}

@app26.get("/health", tags=["health"], include_in_schema=True)
def health():
    return {"status": "ok"}

# ── Custom OpenAPI schema manipulation ────────────────────────
def custom_openapi():
    if app26.openapi_schema:
        return app26.openapi_schema
    schema = get_openapi(
        title=app26.title, version=app26.version,
        description=app26.description, routes=app26.routes
    )
    # Add global security scheme
    schema["components"]["securitySchemes"] = {
        "BearerAuth": {"type": "http", "scheme": "bearer", "bearerFormat": "JWT"}
    }
    schema["security"] = [{"BearerAuth": []}]
    app26.openapi_schema = schema
    return schema

app26.openapi = custom_openapi
print("OpenAPI schema customized. Visit /docs for Swagger UI.")

In [ ]:
server26 = run_app(app26, port=8022)

with httpx.Client(base_url="http://127.0.0.1:8022") as c:
    schema = c.get("/openapi.json").json()
    print("API title:",   schema["info"]["title"])
    print("API version:", schema["info"]["version"])
    print("Security:",    schema.get("security"))
    print("Tags:",        [t["name"] for t in schema.get("tags", [])])
    print("Paths:",       list(schema["paths"].keys()))

---
## 27. Celery Task Queues

Offload long-running tasks (video processing, ML inference, emails) to Celery workers.

In [ ]:
# pip install celery redis
# Requires Redis running: docker run -d -p 6379:6379 redis

# ── celery_app.py ─────────────────────────────────────────────
celery_setup_code = '''
from celery import Celery

celery_app = Celery(
    "tasks",
    broker="redis://localhost:6379/0",    # Task queue
    backend="redis://localhost:6379/1",   # Result storage
    include=["tasks"]
)

celery_app.conf.update(
    task_serializer="json",
    result_serializer="json",
    accept_content=["json"],
    timezone="UTC",
    enable_utc=True,
    task_track_started=True,
    result_expires=3600,
)
'''

# ── tasks.py ──────────────────────────────────────────────────
tasks_code = '''
from celery_app import celery_app
import time, hashlib

@celery_app.task(bind=True, max_retries=3)
def process_video(self, video_id: int, resolution: str):
    """Long-running video processing task."""
    try:
        self.update_state(state="PROGRESS", meta={"progress": 0})
        for i in range(1, 5):
            time.sleep(1)  # Simulate work
            self.update_state(state="PROGRESS", meta={"progress": i * 25})
        return {"video_id": video_id, "resolution": resolution, "status": "done"}
    except Exception as exc:
        raise self.retry(exc=exc, countdown=5)

@celery_app.task
def send_email(to: str, subject: str, body: str):
    time.sleep(0.5)
    return {"sent_to": to, "subject": subject}

@celery_app.task
def compute_hash(data: str) -> str:
    return hashlib.sha256(data.encode()).hexdigest()
'''

# ── main.py — FastAPI + Celery integration ────────────────────
fastapi_celery_code = '''
from fastapi import FastAPI, HTTPException
from celery.result import AsyncResult
from tasks import process_video, send_email, compute_hash

app = FastAPI()

@app.post("/process-video")
def start_video_processing(video_id: int, resolution: str = "1080p"):
    task = process_video.delay(video_id, resolution)  # Non-blocking!
    return {"task_id": task.id, "status": "queued"}

@app.get("/tasks/{task_id}")
def get_task_status(task_id: str):
    result = AsyncResult(task_id)
    return {
        "task_id": task_id,
        "state":   result.state,        # PENDING | STARTED | PROGRESS | SUCCESS | FAILURE
        "info":    result.info,          # Progress metadata or result
        "ready":   result.ready(),
        "result":  result.result if result.ready() else None
    }

@app.post("/email")
def queue_email(to: str, subject: str, body: str):
    task = send_email.apply_async(
        args=[to, subject, body],
        countdown=5,          # Delay 5 seconds
        expires=3600          # Task expires in 1 hour
    )
    return {"task_id": task.id}
'''

print("=== celery_app.py ===", celery_setup_code)
print("=== tasks.py ===", tasks_code)
print("=== main.py ===", fastapi_celery_code)
print("\nStart worker with:")
print("  celery -A celery_app worker --loglevel=info --concurrency=4")
print("Monitor with:")
print("  celery -A celery_app flower  # Web UI at localhost:5555")

---
## 28. Prometheus Metrics & OpenTelemetry

Expose metrics for Prometheus scraping and add distributed tracing.

In [ ]:
# pip install prometheus-fastapi-instrumentator opentelemetry-sdk opentelemetry-instrumentation-fastapi
import time
from fastapi import FastAPI, Request

app28 = FastAPI()

# ── Prometheus via prometheus-fastapi-instrumentator ──────────
try:
    from prometheus_fastapi_instrumentator import Instrumentator
    from prometheus_client import Counter, Histogram, Gauge

    # Auto-instrument all routes
    Instrumentator().instrument(app28).expose(app28, endpoint="/metrics")

    # Custom metrics
    REQUEST_COUNT = Counter(
        "myapp_requests_total",
        "Total number of requests",
        labelnames=["method", "endpoint", "status"]
    )
    LATENCY = Histogram(
        "myapp_request_latency_seconds",
        "Request latency",
        labelnames=["endpoint"]
    )
    ACTIVE_USERS = Gauge("myapp_active_users", "Number of active users")

    @app28.middleware("http")
    async def track_metrics(request: Request, call_next):
        start = time.time()
        response = await call_next(request)
        duration = time.time() - start
        REQUEST_COUNT.labels(request.method, request.url.path, response.status_code).inc()
        LATENCY.labels(request.url.path).observe(duration)
        return response

    print("✅ Prometheus metrics enabled at /metrics")
except ImportError:
    print("Install: pip install prometheus-fastapi-instrumentator prometheus-client")

@app28.get("/")
def root():
    return {"status": "ok"}

@app28.get("/users/{user_id}")
def get_user(user_id: int):
    return {"id": user_id}

In [ ]:
# ── OpenTelemetry tracing ─────────────────────────────────────
otel_code = '''
# pip install opentelemetry-sdk opentelemetry-instrumentation-fastapi
#             opentelemetry-exporter-otlp

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor

# Setup tracing provider
provider = TracerProvider()
exporter = OTLPSpanExporter(endpoint="http://localhost:4317")  # Jaeger / Tempo
provider.add_span_processor(BatchSpanProcessor(exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("myapp")

app = FastAPI()

# Auto-instrument FastAPI
FastAPIInstrumentor.instrument_app(app)

@app.get("/users/{user_id}")
async def get_user(user_id: int):
    # Manual span for custom traces
    with tracer.start_as_current_span("db-query") as span:
        span.set_attribute("db.table", "users")
        span.set_attribute("user.id", user_id)
        await asyncio.sleep(0.01)  # Simulate DB
        return {"id": user_id, "name": "Alice"}
'''
print("OpenTelemetry setup (requires collector running):")
print(otel_code)
print("Run Jaeger locally:")
print("  docker run -d -p 16686:16686 -p 4317:4317 jaegertracing/all-in-one")

---
## 29. GraphQL with Strawberry

Add a full GraphQL endpoint alongside your REST API using Strawberry.

In [ ]:
# pip install strawberry-graphql[fastapi]
try:
    import strawberry
    from strawberry.fastapi import GraphQLRouter
    from typing import List, Optional
    from fastapi import FastAPI

    # ── Strawberry types ───────────────────────────────────────
    @strawberry.type
    class Author:
        id:   int
        name: str

    @strawberry.type
    class Book:
        id:     int
        title:  str
        year:   int
        author: Author

    # Fake data
    AUTHORS = [Author(id=1, name="George Orwell"), Author(id=2, name="Aldous Huxley")]
    BOOKS   = [
        Book(id=1, title="1984",          year=1949, author=AUTHORS[0]),
        Book(id=2, title="Animal Farm",   year=1945, author=AUTHORS[0]),
        Book(id=3, title="Brave New World",year=1932, author=AUTHORS[1]),
    ]

    # ── Query type ─────────────────────────────────────────────
    @strawberry.type
    class Query:
        @strawberry.field
        def books(self) -> List[Book]:
            return BOOKS

        @strawberry.field
        def book(self, id: int) -> Optional[Book]:
            return next((b for b in BOOKS if b.id == id), None)

        @strawberry.field
        def authors(self) -> List[Author]:
            return AUTHORS

    # ── Mutation type ──────────────────────────────────────────
    @strawberry.type
    class Mutation:
        @strawberry.mutation
        def add_book(self, title: str, year: int, author_id: int) -> Book:
            author = next((a for a in AUTHORS if a.id == author_id), AUTHORS[0])
            new_book = Book(id=len(BOOKS) + 1, title=title, year=year, author=author)
            BOOKS.append(new_book)
            return new_book

    # ── Subscription (requires async transport) ────────────────
    import asyncio
    from typing import AsyncGenerator

    @strawberry.type
    class Subscription:
        @strawberry.subscription
        async def count(self, target: int = 5) -> AsyncGenerator[int, None]:
            for i in range(1, target + 1):
                yield i
                await asyncio.sleep(0.5)

    # ── Mount GraphQL on FastAPI ───────────────────────────────
    schema   = strawberry.Schema(query=Query, mutation=Mutation, subscription=Subscription)
    gql_router = GraphQLRouter(schema, graphql_ide="graphiql")

    app29 = FastAPI()
    app29.include_router(gql_router, prefix="/graphql")

    @app29.get("/rest/books")
    def rest_books():
        return [{"id": b.id, "title": b.title} for b in BOOKS]

    print("✅ GraphQL + REST mounted")
    print("   GraphiQL IDE: http://localhost:8023/graphql")

except ImportError:
    print("Install: pip install 'strawberry-graphql[fastapi]'")
    app29 = FastAPI()

    @app29.get("/graphql-placeholder")
    def placeholder():
        return {"note": "Install strawberry-graphql[fastapi] to use GraphQL"}

In [ ]:
server29 = run_app(app29, port=8023)

with httpx.Client(base_url="http://127.0.0.1:8023") as c:
    # Query all books
    query = """
    {
      books { id title year author { name } }
    }
    """
    r = c.post("/graphql", json={"query": query})
    if r.status_code == 200:
        print("GraphQL books:", r.json()["data"]["books"])

        # Mutation
        mutation = 'mutation { addBook(title: "Coming Up for Air", year: 1939, authorId: 1) { id title } }'
        r2 = c.post("/graphql", json={"query": mutation})
        print("Added book:", r2.json()["data"]["addBook"])
    else:
        print("GraphQL not available — install strawberry-graphql[fastapi]")

    print("REST books:", c.get("/rest/books" if r.status_code == 200 else "/graphql-placeholder").json())

---
## 30. Docker Compose & Nginx

Production deployment with Docker Compose: FastAPI + PostgreSQL + Redis + Nginx.

In [ ]:
docker_compose_yml = '''
# docker-compose.yml
version: "3.9"

services:

  # ── FastAPI App ───────────────────────────────────────────
  api:
    build: .
    command: uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4
    environment:
      DATABASE_URL: postgresql+asyncpg://user:pass@db:5432/mydb
      REDIS_URL:    redis://redis:6379/0
      SECRET_KEY:   ${SECRET_KEY}
      ENVIRONMENT:  production
    depends_on:
      db:    { condition: service_healthy }
      redis: { condition: service_started }
    volumes:
      - ./uploads:/app/uploads
    restart: unless-stopped
    networks: [backend]

  # ── PostgreSQL ────────────────────────────────────────────
  db:
    image: postgres:16-alpine
    environment:
      POSTGRES_USER:     user
      POSTGRES_PASSWORD: pass
      POSTGRES_DB:       mydb
    volumes:
      - pgdata:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U user -d mydb"]
      interval: 10s
      timeout: 5s
      retries: 5
    networks: [backend]

  # ── Redis ─────────────────────────────────────────────────
  redis:
    image: redis:7-alpine
    volumes:
      - redisdata:/data
    command: redis-server --appendonly yes
    networks: [backend]

  # ── Celery Worker ─────────────────────────────────────────
  worker:
    build: .
    command: celery -A celery_app worker --loglevel=info --concurrency=4
    environment:
      DATABASE_URL: postgresql+asyncpg://user:pass@db:5432/mydb
      REDIS_URL:    redis://redis:6379/0
    depends_on: [api, redis, db]
    restart: unless-stopped
    networks: [backend]

  # ── Nginx reverse proxy ───────────────────────────────────
  nginx:
    image: nginx:alpine
    ports:
      - "80:80"
      - "443:443"
    volumes:
      - ./nginx/nginx.conf:/etc/nginx/nginx.conf:ro
      - ./nginx/ssl:/etc/nginx/ssl:ro
      - ./static:/var/www/static:ro
    depends_on: [api]
    networks: [backend, frontend]
    restart: unless-stopped

volumes:
  pgdata:
  redisdata:

networks:
  backend:
  frontend:
'''
print(docker_compose_yml)

In [ ]:
nginx_conf = '''
# nginx/nginx.conf
worker_processes auto;

events { worker_connections 1024; }

http {
    # Upstream FastAPI cluster
    upstream api {
        server api:8000;
        keepalive 32;
    }

    # Redirect HTTP → HTTPS
    server {
        listen 80;
        server_name api.example.com;
        return 301 https://$host$request_uri;
    }

    # HTTPS server
    server {
        listen 443 ssl http2;
        server_name api.example.com;

        ssl_certificate     /etc/nginx/ssl/cert.pem;
        ssl_certificate_key /etc/nginx/ssl/key.pem;
        ssl_protocols       TLSv1.2 TLSv1.3;
        ssl_ciphers         HIGH:!aNULL:!MD5;

        # Security headers
        add_header Strict-Transport-Security "max-age=31536000" always;
        add_header X-Frame-Options DENY;
        add_header X-Content-Type-Options nosniff;

        # Proxy to FastAPI
        location /api/ {
            proxy_pass         http://api/;
            proxy_http_version 1.1;
            proxy_set_header   Upgrade $http_upgrade;
            proxy_set_header   Connection "upgrade";  # For WebSocket
            proxy_set_header   Host $host;
            proxy_set_header   X-Real-IP $remote_addr;
            proxy_set_header   X-Forwarded-For $proxy_add_x_forwarded_for;
            proxy_set_header   X-Forwarded-Proto $scheme;
            proxy_read_timeout 300s;  # Long timeout for SSE/streaming
        }

        # Serve static files directly
        location /static/ {
            alias  /var/www/static/;
            expires 30d;
            add_header Cache-Control "public, immutable";
        }

        # Rate limiting
        limit_req_zone $binary_remote_addr zone=api:10m rate=30r/m;
        location /api/auth/ {
            limit_req zone=api burst=5 nodelay;
            proxy_pass http://api/auth/;
        }
    }
}
'''
print("Nginx config:")
print(nginx_conf)

print("\nUseful commands:")
print("  docker compose up -d           # Start all services")
print("  docker compose logs -f api     # Stream API logs")
print("  docker compose scale api=3     # Scale API to 3 instances")
print("  docker compose exec api bash   # Shell into container")
print("  docker compose down -v         # Stop & remove volumes")

---
## 31. gRPC with FastAPI

Run a gRPC server alongside FastAPI — ideal for internal microservice communication with high performance and strong typing via Protocol Buffers.

In [ ]:
# pip install grpcio grpcio-tools grpcio-reflection

# ── Step 1: Define your .proto file ──────────────────────────
proto_content = '''
// user_service.proto
syntax = "proto3";

package userservice;

// ── Messages ──────────────────────────────────────────────
message GetUserRequest  { int32 user_id = 1; }
message CreateUserRequest {
  string name  = 1;
  string email = 2;
  int32  age   = 3;
}
message UserResponse {
  int32  id    = 1;
  string name  = 2;
  string email = 3;
  int32  age   = 4;
}
message UserListResponse { repeated UserResponse users = 1; }
message Empty {}

// ── Service definition ─────────────────────────────────────
service UserService {
  rpc GetUser    (GetUserRequest)    returns (UserResponse);
  rpc CreateUser (CreateUserRequest) returns (UserResponse);
  rpc ListUsers  (Empty)             returns (UserListResponse);
  rpc StreamUsers(Empty)             returns (stream UserResponse); // server-side stream
}
'''

with open("/tmp/user_service.proto", "w") as f:
    f.write(proto_content)

print("Proto file written. Generate Python stubs with:")
print("  python -m grpc_tools.protoc -I. --python_out=. --grpc_python_out=. user_service.proto")

In [ ]:
import subprocess, os

# Generate stubs
result = subprocess.run(
    ["python", "-m", "grpc_tools.protoc",
     "-I/tmp", "--python_out=/tmp", "--grpc_python_out=/tmp",
     "/tmp/user_service.proto"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ Stubs generated:", [f for f in os.listdir("/tmp") if "user_service" in f])
else:
    print("Install grpcio-tools: pip install grpcio-tools")
    print(result.stderr)

In [ ]:
# ── Step 2: Implement the gRPC servicer ──────────────────────
grpc_servicer_code = '''
# grpc_server.py
import grpc, asyncio
import user_service_pb2 as pb2
import user_service_pb2_grpc as pb2_grpc

# Fake DB
USERS = {
    1: {"id": 1, "name": "Alice", "email": "alice@example.com", "age": 30},
    2: {"id": 2, "name": "Bob",   "email": "bob@example.com",   "age": 25},
}

class UserServicer(pb2_grpc.UserServiceServicer):

    async def GetUser(self, request, context):
        user = USERS.get(request.user_id)
        if not user:
            context.set_code(grpc.StatusCode.NOT_FOUND)
            context.set_details(f"User {request.user_id} not found")
            return pb2.UserResponse()
        return pb2.UserResponse(**user)

    async def CreateUser(self, request, context):
        new_id = max(USERS) + 1
        user = {"id": new_id, "name": request.name,
                "email": request.email, "age": request.age}
        USERS[new_id] = user
        return pb2.UserResponse(**user)

    async def ListUsers(self, request, context):
        return pb2.UserListResponse(users=[pb2.UserResponse(**u) for u in USERS.values()])

    async def StreamUsers(self, request, context):  # Server-side streaming
        for user in USERS.values():
            yield pb2.UserResponse(**user)
            await asyncio.sleep(0.1)

async def serve_grpc():
    server = grpc.aio.server()
    pb2_grpc.add_UserServiceServicer_to_server(UserServicer(), server)
    server.add_insecure_port("[::]:50051")
    await server.start()
    print("gRPC server started on port 50051")
    await server.wait_for_termination()
'''
print(grpc_servicer_code)

In [ ]:
# ── Step 3: Run gRPC + FastAPI side by side ───────────────────
combined_code = '''
# main.py — FastAPI REST + gRPC running concurrently
import asyncio
from contextlib import asynccontextmanager
from fastapi import FastAPI
import grpc
import user_service_pb2       as pb2
import user_service_pb2_grpc  as pb2_grpc
from grpc_server import UserServicer, serve_grpc

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Start gRPC server in background
    grpc_task = asyncio.create_task(serve_grpc())
    yield
    grpc_task.cancel()

app = FastAPI(lifespan=lifespan)

# ── gRPC client helper (reusable channel) ─────────────────────
def get_grpc_stub():
    channel = grpc.aio.insecure_channel("localhost:50051")
    return pb2_grpc.UserServiceStub(channel)

# ── REST endpoints that call gRPC internally ──────────────────
@app.get("/users/{user_id}")
async def get_user(user_id: int):
    stub     = get_grpc_stub()
    response = await stub.GetUser(pb2.GetUserRequest(user_id=user_id))
    return {"id": response.id, "name": response.name, "email": response.email}

@app.get("/users")
async def list_users():
    stub     = get_grpc_stub()
    response = await stub.ListUsers(pb2.Empty())
    return [{"id": u.id, "name": u.name} for u in response.users]

@app.post("/users")
async def create_user(name: str, email: str, age: int):
    stub     = get_grpc_stub()
    response = await stub.CreateUser(
        pb2.CreateUserRequest(name=name, email=email, age=age)
    )
    return {"id": response.id, "name": response.name}
'''
print(combined_code)
print("\n📌 Architecture:")
print("  Browser / external clients  →  REST (port 8000)  →  FastAPI")
print("  Internal microservices      →  gRPC (port 50051) →  Servicer")
print("  FastAPI can also call gRPC internally as a client")

In [ ]:
# ── gRPC client usage example ─────────────────────────────────
grpc_client_code = '''
# grpc_client.py — test the gRPC server directly
import grpc
import asyncio
import user_service_pb2      as pb2
import user_service_pb2_grpc as pb2_grpc

async def main():
    async with grpc.aio.insecure_channel("localhost:50051") as channel:
        stub = pb2_grpc.UserServiceStub(channel)

        # Unary call
        user = await stub.GetUser(pb2.GetUserRequest(user_id=1))
        print("GetUser:", user.name, user.email)

        # Create user
        new = await stub.CreateUser(
            pb2.CreateUserRequest(name="Carol", email="carol@x.com", age=28)
        )
        print("Created:", new.id, new.name)

        # Server-side streaming
        print("Streaming users:")
        async for u in stub.StreamUsers(pb2.Empty()):
            print(" -", u.name)

asyncio.run(main())
'''
print(grpc_client_code)

---
## 32. Multi-tenancy Patterns

Serve multiple isolated tenants from a single FastAPI deployment — three common strategies.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Header, Request
from pydantic import BaseModel
from typing import Optional

app32 = FastAPI()

# ── Tenant registry (would come from DB in production) ────────
TENANTS = {
    "tenant-abc": {"id": "tenant-abc", "name": "Acme Corp",   "plan": "enterprise", "db": "db_acme"},
    "tenant-xyz": {"id": "tenant-xyz", "name": "Startup Ltd", "plan": "starter",    "db": "db_startup"},
    "tenant-def": {"id": "tenant-def", "name": "BigCo Inc",   "plan": "pro",        "db": "db_bigco"},
}

# ── Strategy 1: Tenant via HTTP Header (X-Tenant-ID) ─────────
def get_tenant_from_header(x_tenant_id: Optional[str] = Header(default=None)):
    if not x_tenant_id:
        raise HTTPException(status_code=400, detail="X-Tenant-ID header is required")
    tenant = TENANTS.get(x_tenant_id)
    if not tenant:
        raise HTTPException(status_code=404, detail=f"Tenant '{x_tenant_id}' not found")
    return tenant

@app32.get("/header/data")
def data_via_header(tenant: dict = Depends(get_tenant_from_header)):
    return {"tenant": tenant["name"], "plan": tenant["plan"], "db": tenant["db"]}

# ── Strategy 2: Tenant via Subdomain (host header) ────────────
def get_tenant_from_host(request: Request):
    host = request.headers.get("host", "")
    # e.g. acme.myapp.com → subdomain = acme
    subdomain = host.split(".")[0] if "." in host else None
    tenant_id_map = {"acme": "tenant-abc", "startup": "tenant-xyz"}
    tenant_id = tenant_id_map.get(subdomain)
    if not tenant_id:
        raise HTTPException(status_code=404, detail=f"No tenant for subdomain: {subdomain}")
    return TENANTS[tenant_id]

@app32.get("/subdomain/data")
def data_via_subdomain(tenant: dict = Depends(get_tenant_from_host)):
    return {"tenant": tenant["name"], "strategy": "subdomain"}

# ── Strategy 3: Tenant via URL path prefix ────────────────────
@app32.get("/t/{tenant_id}/data")
def data_via_path(tenant_id: str):
    tenant = TENANTS.get(tenant_id)
    if not tenant:
        raise HTTPException(status_code=404, detail="Tenant not found")
    return {"tenant": tenant["name"], "strategy": "path-prefix"}

In [ ]:
# ── Per-tenant DB session (schema-per-tenant or DB-per-tenant) ─
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker, Session

# DB-per-tenant: each tenant has their own SQLite file
_tenant_engines = {}

def get_tenant_engine(tenant_id: str):
    if tenant_id not in _tenant_engines:
        db_url = f"sqlite:////tmp/tenant_{tenant_id}.db"
        engine = create_engine(db_url, connect_args={"check_same_thread": False})
        _tenant_engines[tenant_id] = engine
    return _tenant_engines[tenant_id]

def get_tenant_db(tenant: dict = Depends(get_tenant_from_header)):
    engine = get_tenant_engine(tenant["id"])
    SessionLocal = sessionmaker(bind=engine)
    db = SessionLocal()
    try:
        yield db, tenant
    finally:
        db.close()

@app32.get("/header/db-info")
def tenant_db_info(ctx: tuple = Depends(get_tenant_db)):
    db, tenant = ctx
    # Each tenant query runs on their isolated DB
    return {"tenant": tenant["name"], "db": tenant["db"], "isolated": True}

In [ ]:
# ── Tenant middleware: attach tenant to request state globally ─
from starlette.middleware.base import BaseHTTPMiddleware

class TenantMiddleware(BaseHTTPMiddleware):
    """Resolves and attaches tenant to every request automatically."""
    EXCLUDED = {"/health", "/docs", "/openapi.json"}

    async def dispatch(self, request: Request, call_next):
        if request.url.path not in self.EXCLUDED:
            tenant_id = request.headers.get("X-Tenant-ID")
            if tenant_id and tenant_id in TENANTS:
                request.state.tenant = TENANTS[tenant_id]
            else:
                request.state.tenant = None
        return await call_next(request)

app32.add_middleware(TenantMiddleware)

# Access tenant from request.state anywhere
@app32.get("/middleware/tenant")
def tenant_from_state(request: Request):
    tenant = getattr(request.state, "tenant", None)
    if not tenant:
        return {"error": "No tenant resolved"}
    return {"tenant": tenant["name"], "plan": tenant["plan"]}

In [ ]:
# ── Tenant-aware feature flags & plan enforcement ─────────────
PLAN_LIMITS = {
    "starter":    {"max_users": 10,  "api_calls_per_day": 1_000,  "features": ["basic"]},
    "pro":        {"max_users": 100, "api_calls_per_day": 50_000,  "features": ["basic", "analytics"]},
    "enterprise": {"max_users": -1,  "api_calls_per_day": -1,      "features": ["basic", "analytics", "sso", "audit"]},
}

def require_feature(feature: str):
    """Dependency factory: enforce feature availability by plan."""
    def check(tenant: dict = Depends(get_tenant_from_header)):
        plan    = tenant["plan"]
        allowed = PLAN_LIMITS.get(plan, {}).get("features", [])
        if feature not in allowed:
            raise HTTPException(
                status_code=403,
                detail=f"Feature '{feature}' not available on plan '{plan}'. Upgrade to access."
            )
        return tenant
    return check

@app32.get("/analytics/report")
def analytics_report(tenant: dict = Depends(require_feature("analytics"))):
    return {"tenant": tenant["name"], "report": {"visits": 12450, "conversions": 320}}

@app32.get("/sso/config")
def sso_config(tenant: dict = Depends(require_feature("sso"))):
    return {"tenant": tenant["name"], "sso": {"provider": "okta", "enabled": True}}

@app32.get("/plan/limits")
def plan_limits(tenant: dict = Depends(get_tenant_from_header)):
    limits = PLAN_LIMITS[tenant["plan"]]
    return {"tenant": tenant["name"], "plan": tenant["plan"], "limits": limits}

In [ ]:
import httpx, time

def run_app(app, port):
    import threading, uvicorn
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="error")
    server = uvicorn.Server(config)
    threading.Thread(target=server.run, daemon=True).start()
    time.sleep(1)
    return server

server32 = run_app(app32, port=8024)

with httpx.Client(base_url="http://127.0.0.1:8024") as c:
    # Strategy 1: header-based tenant
    r = c.get("/header/data", headers={"X-Tenant-ID": "tenant-abc"})
    print("Header tenant:", r.json())

    # Strategy 3: path-based tenant
    r = c.get("/t/tenant-xyz/data")
    print("Path tenant:",   r.json())

    # Plan limits
    r = c.get("/plan/limits", headers={"X-Tenant-ID": "tenant-xyz"})
    print("Plan limits:",   r.json())

    # Feature enforcement: starter plan trying to access analytics
    r = c.get("/analytics/report", headers={"X-Tenant-ID": "tenant-xyz"})
    print("Starter → analytics (blocked):", r.status_code, r.json()["detail"])

    # Enterprise plan accessing analytics
    r = c.get("/analytics/report", headers={"X-Tenant-ID": "tenant-abc"})
    print("Enterprise → analytics (allowed):", r.json())

    # Middleware-resolved tenant
    r = c.get("/middleware/tenant", headers={"X-Tenant-ID": "tenant-def"})
    print("Middleware tenant:", r.json())

---
## 🎯 Quick Reference Summary

| Module | Key Classes / Functions | Use Case |
|--------|------------------------|----------|
| **Routing** | `@app.get/post/put/patch/delete` | Define HTTP endpoints |
| **Path & Query Params** | `Path()`, `Query()`, `Enum` | URL params with validation |
| **Request Body** | `BaseModel`, `Field`, `@field_validator` | Validate JSON bodies |
| **Response Models** | `response_model=`, `status_code=`, `JSONResponse` | Control output & status |
| **Headers/Cookies** | `Header()`, `Cookie()`, `Form()` | Read/set headers & cookies |
| **File Uploads** | `UploadFile`, `File()` | Handle file uploads |
| **Dependency Injection** | `Depends()`, `yield` deps | Shared logic & resources |
| **Auth & Security** | `OAuth2PasswordBearer`, `jwt`, `APIKeyHeader` | JWT, API key auth |
| **Database** | SQLAlchemy + `get_db()` dep | ORM with session management |
| **Async SQLAlchemy** | `AsyncSession`, `async_sessionmaker` | Non-blocking DB access |
| **Async** | `async def`, `await`, `StreamingResponse` | Non-blocking I/O |
| **Background Tasks** | `BackgroundTasks.add_task()` | Post-response async jobs |
| **Middleware** | `@app.middleware`, `BaseHTTPMiddleware` | Cross-cutting concerns |
| **CORS** | `CORSMiddleware` | Cross-origin browser access |
| **WebSockets** | `WebSocket`, `WebSocketDisconnect` | Real-time communication |
| **SSE** | `StreamingResponse`, `text/event-stream` | Server-push streams |
| **Error Handling** | `@app.exception_handler`, custom exceptions | Consistent error responses |
| **Routers** | `APIRouter`, `include_router()` | App modularization |
| **Testing** | `TestClient`, `dependency_overrides` | Unit & integration tests |
| **Lifespan** | `@asynccontextmanager lifespan` | Startup/shutdown hooks |
| **Pagination** | offset, cursor, keyset patterns | Scalable list endpoints |
| **Static/Templates** | `StaticFiles`, `Jinja2Templates` | HTML pages & assets |
| **OpenAPI** | `get_openapi()`, `tags_metadata`, custom docs | Docs & schema control |
| **Caching** | TTL cache, `lru_cache` | Reduce repeated computation |
| **Rate Limiting** | Custom limiter / `slowapi` | Throttle API access |
| **Celery** | `task.delay()`, `AsyncResult` | Background task queues |
| **Metrics** | `Instrumentator`, `Counter`, `Histogram` | Prometheus observability |
| **Tracing** | `FastAPIInstrumentor`, `tracer.start_as_current_span` | OpenTelemetry distributed tracing |
| **GraphQL** | `strawberry.Schema`, `GraphQLRouter` | GraphQL endpoint |
| **gRPC** | `grpc.aio.server`, `Servicer`, proto stubs | High-perf microservice RPC |
| **Multi-tenancy** | Header/subdomain/path strategies, `require_feature()` | Isolated per-tenant data & plans |

---
### 📚 Resources
- [FastAPI Official Docs](https://fastapi.tiangolo.com)
- [FastAPI GitHub](https://github.com/tiangolo/fastapi)
- [Pydantic v2 Docs](https://docs.pydantic.dev)
- [SQLAlchemy Docs](https://docs.sqlalchemy.org)
- [Uvicorn Docs](https://www.uvicorn.org)
- [Starlette Docs](https://www.starlette.io)